# 🏥 Medical VLM Decisive Validation Experiment (Kaggle Dual-T4 GPU) — Revision rev02
**Research Project:** Evaluasi Reliabilitas dan Selective Prediction pada Medical Vision-Language Model di Bawah Pergeseran Bahasa (IndoRad-VQA)  
**Author:** Imam (Undergraduate CS Thesis)  
**Revision:** `rev02` (Strict Manifest Enforcement & Zero-Leakage Test-Only Split)  
**Environment:** Kaggle Dual Tesla T4 (2x 16GB VRAM, 4-bit NF4 Quantization)  

---

## 📌 Metodologi & Arahan Resmi Dosen Pembimbing
1. **Peniadaan Blur Total:** Sesuai arahan dosen pembimbing, manipulasi blur dieliminasi secara permanen. Hanya dua kondisi visual yang dievaluasi:
   - `P0_ORIGINAL`: Citra radiologi asli tanpa manipulasi.
   - `P1_BLACK_BLANK`: Kanvas hitam murni `RGB (0, 0, 0)` berdimensi dan mode identik dengan citra asli.
2. **Target Candidate Sequence Scoring:** Menghindari *single-token logit hacking*. Menghitung *conditional sequence log-likelihood* secara eksak dengan runtime prefix-match assertion untuk:
   - **English:** `["Yes", "No"]`
   - **Indonesian:** `["Ya", "Tidak"]`
3. **Komparasi Fair 5 Estimator:**
   - $M_1$: Confidence-only baseline ($p_{\text{orig}}$)
   - $M_2$: Visual-only ($\max(0, \Delta_{\text{dir}})$)
   - $M_{\text{dir}}$: Directed Multiplicative ($p_{\text{orig}} \times \max(0, \Delta_{\text{dir}})$)
   - $M_{\text{norm}}$: Normalized Directed Linear ($p_{\text{orig}} \times \frac{1 + \Delta_{\text{dir}}}{2}$)
   - $M_3^{\text{old}}$: Historical Unsigned Baseline ($p_{\text{orig}} \times \frac{1 + |\Delta_{\text{dir}}|}{2}$)
4. **Disiplin Split Data (Zero Leakage, Authoritative Frozen Manifest):**
   - **Primary Test-Only Subset:** $N = 92$ pasangan QA (35 klaster citra fisik unik). Evaluasi utama bebas dari data dev.
   - **Full Benchmark Sensitivity Set:** $N = 150$ pasangan QA (58 klaster citra fisik unik). Memuat split test (92) + dev (58) untuk uji sensitivitas ukuran sampel.
   - **Bit-Identical Manifest Verification:** Manifest divalidasi langsung terhadap SHA256 otoritatif (`988f57a6512223337bb5ab3b192f1802e38bfec2f6b262f51607381faa74bf52`).
   - Tidak ada re-downloading dari Hugging Face atau pembuatan split ad-hoc.
5. **Isolasi Revisi (`rev02`):**
   - Seluruh luaran disimpan di direktori terpisah: `results/decisive_validation/rev02/`.
   - Checkpoint lama yang tidak kompatibel diidentifikasi dan diisolasi secara otomatis.

In [1]:
# ==============================================================================
# CELL 1: ENVIRONMENT & HARDWARE AUDIT
# ==============================================================================
import os
import sys
import subprocess
import platform

# Ensure UTF-8 stdout encoding in any terminal or Jupyter kernel
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Robust display fallback
try:
    from IPython.display import display, FileLink
except ImportError:
    display = print
    FileLink = str

print("=" * 70)
print("🔍 SYSTEM & HARDWARE AUDIT")
print("=" * 70)
print(f"OS Platform     : {platform.platform()}")
print(f"Python Version  : {sys.version.split()[0]}")

# Ensure critical libraries are installed
required_pkgs = ["transformers", "accelerate", "bitsandbytes", "qwen_vl_utils", "torch", "huggingface_hub", "pyarrow"]
missing_pkgs = []
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"📦 Installing missing packages: {missing_pkgs}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs)
    print("✅ Installation complete.")

import torch
import transformers

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"Device Count    : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f"  - GPU {i}: {props.name} ({vram_gb:.2f} GB VRAM, Compute {props.major}.{props.minor})")
else:
    print("⚠️ WARNING: Running on CPU! GPU acceleration strongly recommended.")
print("=" * 70)

🔍 SYSTEM & HARDWARE AUDIT
OS Platform     : Linux-6.18.48+-x86_64-with-glibc2.39
Python Version  : 3.13.15
📦 Installing missing packages: ['bitsandbytes', 'qwen_vl_utils']...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 63.5 MB/s eta 0:00:00
✅ Installation complete.
PyTorch Version : 2.11.0+cu128
CUDA Available  : True
CUDA Version    : 12.8
Device Count    : 2
  - GPU 0: Tesla T4 (14.56 GB VRAM, Compute 7.5)
  - GPU 1: Tesla T4 (14.56 GB VRAM, Compute 7.5)


In [2]:
# ==============================================================================
# CELL 2: REVISION REV02 PATHS & ISOLATION
# ==============================================================================
from pathlib import Path
from datetime import datetime, timezone
import json

REVISION_ID = "rev02"
AUTHORITATIVE_MANIFEST_SHA256 = "988f57a6512223337bb5ab3b192f1802e38bfec2f6b262f51607381faa74bf52"

# Workspace setup
WORKSPACE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
PROJECT_DIR = WORKSPACE / "indorad_validation_rev02"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = DATA_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Revision-specific output directory
RESULTS_DIR = WORKSPACE / "results" / "decisive_validation" / "rev02"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PERTURB_DIR = DATA_DIR / "scratch_perturbations_rev02"
PERTURB_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Active Revision    : {REVISION_ID}")
print(f"📂 Project Root       : {PROJECT_DIR.resolve()}")
print(f"📂 Data Directory     : {DATA_DIR.resolve()}")
print(f"📂 Results Directory  : {RESULTS_DIR.resolve()}")
print(f"📂 Perturbations Dir  : {PERTURB_DIR.resolve()}")

✅ Active Revision    : rev02
📂 Project Root       : /kaggle/working/indorad_validation_rev02
📂 Data Directory     : /kaggle/working/indorad_validation_rev02/data
📂 Results Directory  : /kaggle/working/results/decisive_validation/rev02
📂 Perturbations Dir  : /kaggle/working/indorad_validation_rev02/data/scratch_perturbations_rev02


In [3]:
# ==============================================================================
# CELL 3: DATASET UNZIPPING & SEARCH
# ==============================================================================
import zipfile

# Search for any dataset zip across Kaggle input directories
zip_candidates = []
if Path("/kaggle/input").exists():
    zip_candidates.extend(list(Path("/kaggle/input").rglob("*.zip")))
if Path("main/Kaggle/indorad-vqa-research.zip").exists():
    zip_candidates.append(Path("main/Kaggle/indorad-vqa-research.zip"))

if zip_candidates:
    for src_zip in zip_candidates:
        print(f"📦 Extracting dataset zip: {src_zip} -> {DATA_DIR}...")
        try:
            with zipfile.ZipFile(src_zip, 'r') as zf:
                zf.extractall(DATA_DIR)
            print(f"✅ Unzipped {src_zip.name} successfully.")
        except Exception as e:
            print(f"⚠️ Could not extract {src_zip}: {e}")
else:
    print("ℹ️ No zip archives found in input. Using existing disk directories.")

ℹ️ No zip archives found in input. Using existing disk directories.


In [4]:
# ==============================================================================
# CELL 4: AUTHORITATIVE FROZEN MANIFEST ENFORCEMENT & INTEGRITY AUDIT
# ==============================================================================
import io
import zlib
import base64
import hashlib
import numpy as np
import pandas as pd

AUTHORITATIVE_MANIFEST_SHA256 = "988f57a6512223337bb5ab3b192f1802e38bfec2f6b262f51607381faa74bf52"
EMBEDDED_MANIFEST_B64 = "eNrNfV2P60iS3XsD/R/0ZhuotvP742kwHix2F7bXxu68+KmRn1XsUkk1onRv1/56nyCTEkVK1dW9Y0AP3eQtUUryMDLiRGRE5N/Cz11+6t7Cc6GT/n3bHZ/+dir9sdvvfi67yzk+PZRaDmWXys9h138vh6fjASfbMHzeH8Px1D91/c8fpf95t39KYbffdSlsfz7//HiyC2+lnX7v8vGlnb+U7vnl2P7xts/l6RC+/9y/BKHN03v3a9m2f/z4g7K47PlnHHL59hQOZdMfD6d0PB1Kvwl9v09dOJa8+d4dXzbHl7J563I8hG632e7T8AlOjy9dvxlG+9PTn9/Da3jZhByGX3rFL20+wu55cyyH19AdN7nsnsPu8jvTh9tyDK+b3G2ew1sMB/xu96en//sP//b0l//5D3/+l6e/Hk7lyXCnc8xFyZhiYmm4d8aYsv/1fff8pPF4Ruqnf/3H//4kPY9eC66cqlIW73Q1TMRsbbUqCGYUE5o5afFlaZPNJXnDNGPWKcdKWI0lVFWFuWy1YTxGpTkvIRamYw06F5OzDVVWWZKLKiT54w/aDfeHA2ELiK7w6wDEsavdDMK+JBKAM4gxPHeA6kiIvWz2hM9/nr7/Xwiz6TdeQ99tctiGt+k7vwc97R4RPSuG+8NhkswBvnB47nb9Zl+Hf6ZwyF1Im77bvuxP5Xgsm93+8Ba2MwyPof9p+D9dhWs2v4Td8QShm678l/89R0laEwTnVnhdBPNhQsmKASXOhBr/Rzi5yEJKnNmMew8JX5GWV1FdkapYKZ3USjhVBGcpe+4qrtJcxJqi8Ebo1WjMMi6cCanIIqWvORcdQuA5e2+81EYVGQW+5bWy1ivgJEec5FzKXko4HDdxf8jlcBORPwqCfFAQ1AiCejpCyY4o4H+YCOHXLmw379uwKxMAmBukn7aERZdJ/YTXviMQFhMmeisz7r0mXVLN8oyCaigw/8QFYwMKNgTIda5RQ9yLsaVKLhIrimuhlBbG1FSDKdzawCLzNebquJY2RGmzCKvRQvUp86qZKpVF5ZMOXEXPVBaS21qjjQBZWqWYFlGzH3/gng23SMcLDjA0m7A5dHFTD2HQ7Fda+h3iANV+2hIQh1N/el0KgwhScS65sdpi0uYGA0a5CINTYkCB3l2xvuCZ8d5Cxt0aLnQWtbhqqk6W410LxUyAdNgqvSiSc5OMr1o7tRrM+RBTzk5U67O0gmteopHOGmOChbiloJxMwYgUgzOOUJANhfmUIBB2m1H1xm0p+RqEcsjhEF5wCSwQadolCFnhRh2vuHPctotnEGRTnvrJ+IaBrw54WC2zqtzrLIspWlurIfY5+iQhM0mkXEsQVSmvcKHOSXKVZCpsNZbS0jmRCaUcHDfVFBNtKilqHU0MgCamACyKZbFw/eMPgvHhBul4LQnPh/Lx0/eX7ki6FPrygAeuAxk5duHK+jRcYsmB5tHuCIDoOzA7mxBPP+E/2Jzd9Kf307F7WU0hHUQQsjiGZ/V4yQ023NcIGxfDf8MEElZEhvfqTaoy4o3C6EBWXA3BARuZYWEitAOz3ljJZfVRVw1TxYVRJq3GchBCwGxomgQSSctj4hXQyZpYdgr6p3pTIl4PC1kSbL7B5hewwTIDnbTZH4BEOR0+emjNTdltYY/KG8CbLBJEdv/W7fBh2B+O4YzmkSQMcw2Qbgv4DYArr2B6MNyYgKfxVzv8KiYkgB++fPmxpTQ6zJDKITNZV6jecIbVz/QzMBuArcAVT1gIn+KgkBxEDrNLJF+TdzUGprzVKmlolapFhZJhhuRLW5ETX41WFVQ4IWwdqSdtTSnVJEgyME34Q4xQYUI7TM4A4w5gxWjN6TgDttmpBuNlSk7G6a28xdKHw0qsrMo+Z6IaQgVVz2Ilroy0GzVzZnjRVpUIWUhSZg/lgWfJyRjOkzIRcysGmySkSONP0hfDU8qpKheMX49WnRA2ymSCrsFyWKkgAUbgFtM7Y24n5yWQSGBDBVYLzy9HLkjHmZlOEK1YttvTG7hf2W32jQiORmtJB2dXE1/uXkiahlm6P+4HUj3asZH8XQtMqsxxvK7grIY+LRNg0l3UF3QFwRWhqUw2IqiYZYre5VKhVwIzMNcB8GSnOVih1M6D0UGNe6jiALNkuWRarsZiYH2Ys8rCfGXQRlyj8e1gEoCtFRpOG+1hDqHtnVEVcKnRkNHxeh7G/a6MfBl8cP8+zbq189EMGrR5Y8QD6xl9ixU4OYhM5tlwVnic7DxGH8Fh4gmPO4ADjS+rgjmysDkV6rsUCeOMaQCZwwzhFqopZC+DNy7zYFQkG4HLDM0iuRoL1h0aHShmV4SsxJthGzy4kKE5iFMdrDMV0LAE5ABOY8aiUeMzNmm/6/dbMBqSmgmkLeZRf4XM63QZVM8oPuFw+on+t8RFgcFEq3gyMqlQ9ITLnApPhj/hl3gEu1cgcCDznGcF0+1B5AKvhcx8shWTDZIkjbAGJpKDwukQq4eorQYD+ckxVWd0yQw+iLQwbfi+J+0NVIzGe7GgDj6D/AQOYFzT3m6pvaGE/naCmQIo/Xt3WNu5dHo9vW+glJ/7QDNppXCY9hIkrXibMTSrExRupnAdG50nWCeNieNcFqVAVXDYbxWtVWSm8XdoDV08I10hoIErtBbXYCQwgEblkFaDMReiAeGDBjIKMlPAocCmwIbAjRT0ui6QwRgD4edcARR+fFl0XDLBuN3v8yZt98crCJ5Pb+/gxCBARIRWABSYBFHIMoDdQhYnALweAFBCPkF7js8PogZdaaAZYzYgtE56k1UuQltw1SqgaKEJoHJV0Hh7Thon4B1CxIuE17kaC78BNWxVKhFTJYoCVQNq7XiAayrL4JBqcEQnS7GOwR+Qjf/Ia/5z9hcxVY770+Qa0VzpX8L7RYNgjlDg4tpJmjxsWPvT6+4jrB1skeFM8+JUIC/HNYxkIzvNKrPRKitfOLwdPLkwpEYxt7MPHiq1iGAznGEXc2bOhQIlakFkIBNgfPiKMiWw1WjRe24iLwXOFuyuji6BT8XKYIZAs4PhUCTwtVnwwlaySpLJhpK8Quk97NKhQF2WXMDu9qeLAsFHr8NHbxSUABzQIWSkd8+v+PzGvAkMHn3SDPJvIchnSOSZ/0k3ig2XeKvC43qXMB8MpINByHNyQkt4TtJI+EHWVAXHAWqDIMOzgQXDtoCRrMYStsjAHCQKVhjKB3DEEhKsEtRIBU/2AladzDyXMQoBQMToR9LxrFpHP3KwMJO5OVB8DX/Nm22pR1jjt/f9AQKy3YOu/GnuZ7ZgFj449ZfrgNQw0Xab1+7w5UAN7uoRAzVSjvaIjnMpegs9JGgHnyp1u+czKvTnQASmL/F0JDk67X45vQISSNKuI0E6kiAtfFARhdYlgGzhxcazHEkxc79GSJQ0hsGldpgDkAU4B5lV4rGVecGSZGAjMM0OwgT3wksQW9A+OAHWgiYHvxoLvM6qykJI2eVY4JJpDeeevNQMexYAPTl0PkbjdCpQv6BEIyRqaYlI8RxCf9yc+inwRz7D/tstDvM6XjyGSHP3fNrRHGtqqElWHvyHfiVDHARFZF0sXB7F2WS7cUcXXWTZSGoEr14ocjUHjSLB7h0cLIFpAr3uvY8ONizDcfXgg9XCbhlnoVAUtDBm6GowqCsQGiGSFbUK7p0HcYyMQfdwGCr4c4oLyCS5/g60Eojp8Q7p+AliZ3guaDSQlgJDnqC2OsGhtEKKKWiB3x+eXxKn427kdBZvtTj4zyBaEcYWtw0nMRYYXK1KhdcDxhJxhBwJKaGgaoJqpLBMhre5Gstyg5kHpldsjZZkKWP2QJnjkSW5rMYl/GaFJw8rRvYa03h8fN/s1RTvTPseNPcF7jmcT4jBtvRkqg7vZyT6U8Y0urruGH4Jb1+2Tv5BrRM017g2wcRCJHb7TflGse80aOQIjnuE4jgHhdeYXdxw8N7XYXLF0+txiAPCkl/ZsDmilX7k9atQ4k4fFEoxihcdya7lfRl19Np8bcL7O5zyZdB4acBgtiBmYIyvdyLIdxxO3MEjOpyqxSfoeB0zBET93J1aAXalseliUsk34FoJ0V2ExGMiZBpC5poabfrw/Nwd8ZRjcHUZtZhF289ximEpCxdSRBFf747HtQDlykPWEfKsYL9rPsNjzvBoM84wKO0sojHViYT7rtyD9kFda+Ek5iQsfnaeHAYoZA2fPRRynmwCbwIFyGw1FsuSeQ36CPoDwpOrq0VXZmEobeUwhhZEycAXgUWEP0/rqKItpAp7D55vXfk+qagBqlsQ0ZzqyAsbYBl8jsOdgPQ9fOxD4tMcUjU5pE0FAaI0mPZN9+1i6seQ8kz9/OWvmz6Ny5qD57Ew/vg7Dt/KLixRuhN/Vs1tfbD4s/KuoeSWbvvu44Zdu5i+90Pp4ZcugvdvEd7r66m/NmrtJ5ZQxeilixRZgbOtLkbNuyujZkdnTYHrOmMZ59BCIcCJU3jnePe6Vs+r8JbjqV1JIIOZYGVVZimrhUhoHVejeags8Gu4IBAZuPsgoYnz5DMrFgehk7EUUzM2Wwe6TWD5BtaSZI9gjOTgN4BbATY49zfxapkKp7f30/bL2PnHxE7zUV3R8XrFFOZ8d56H6XilozAL4Zq9hwvnJpP2T//8j//087/+87/9j0+JN0Z6ROKtm9+hZ37HEoNBY9fugH+Qj9YN+Sr4R6XFIAhK/w6TB2/u1uLiErKznof2g7ubOvxx8HwP0PxhcyLpm69GDpay+X/pdOieQ/e7UH9Md4cM04C6UVfuTlN0IY6csjt25O7QykijX/070L0O3b6WLWAPs0Siy8LItgNDXc7VO54x7uUhPWPdYq10nGXCEFQNJsjmLkMq+/P60Y3lkPJ2okWQ6SsDPJ9kWt1Z7sZNPOJyt27uMx3n7KsxUmIOC38Gqh6ghW6aYqTebjsy95Dw/DGRMA0J02YW5hN0THopRCbhNo+rQ7f5wu6DJsxA0fGd1xKuJ9UXrR7GflCr55vV8/YGvZqzhjVi95EaptII15cBso8JkGm0gI6zZWrKGflvVzkjLbPse9lur/JHZpZvTDP7WtrIRW3/Am3efzUgbvhD5n0aMS6r0HGRRTKLWr3AzTvMM0iuJGyMTr23/AfAhcuX0nVnTRLDPuSaJGX6jbAsQ7zduNwECrUdsj7C4XUwZzsKRA1pIt329LacgdNSAdElCuOd5+J5PZsiMbngoukHgN+CN93FUD8ohq5h6K7SYl+7vCsf/dLKgav+cl63BAPNXX8xc8sFXpkSVFM2ifsiZDwj4Vae8hBbdMzTvaekKPkQ+sZgjsmCR4XrqyOTEerKBBVczFEaOEPaASLhpVyNJYzw3lXmmas2YBZSTlGK0jGazM7wLGSSUoCFwcV2nIDwDQg/B6Kf8n832/2ehGhCZgHJTw0ZMvtn3fN7gfEPCYxpEmKWIYSetPUsTE5avenztRc8Kp1hOv0SDt0wua6+sFxu4q4YEeAqgPxqM6XQ4C5GkKR6MsaM6cJwOxzILayUjcJmWCIS+0K5NDw4WT3Ij1SMp1xq9Jwl65OvQUlGq3duNZbUpnKhlTQ+FZWsdF7GyI2iBV6XMrRyKlzBk7GWW50BUuOLZsEXwzkh4IpAzxZyBz50myhybj1TUcIVwCTOk7E3M6JotR8z0uA/WFVDgnchRC6aNIuh4BpUhi/a2SSiSomUBEV+XVJJC4Fn0cnLIlZjKdCDXMAkLIy+hOCYAIFgFT4GjH4R+MlSKVZXaSXdQk5sW12h48zU32fML6Hvtive/LugsEw8JBSyZY9LtQz+t0wIyvx878sp71P/0S9Y8/D3164/hnN2xDqdninMbxgFkwX0woSHHF1OEJAnK8Zkxaor55TgbnzkcCBAUZLiQC7h8ZSgGCUc66hDghIR8CRzgjUKIIPGZWia1Vg1c61ycppyqlwBs9E111hKYiJLTJGQcizJBimtDDUSHqbhYVZ4jCB8UChkd5UrcheT3P1+WMwjwgLJGm6QjldrtEOK3SZtSzjM8mQmDrKkttcQ3ElkxRiPmcjq2lui4yJyM8JQD2Wgt1fJiIuCgwma0bkcVmTnaSDgdNdpijcjXneRE4+KnGzILb2CCuHoqQAwrwPSOwo3Eyjb8rZeA7pToIIhHrNAxSnVMLgU6lw9/c24xCAPP41SQfanHyKjvw+QB63YcXpUtXSc507FPcgaVGd/LLs8c6pP/bCs3DTK2zlr4as4aPOoOLiGw4W1zjHo78eszmSVsPkyDu5BcXBNSTh5vS5ze9F4XBf+GhXHTz4iFfd8nAF0XCycH0/54zcXzKEiCy0UjHWyR7Iot1bL7zhxGPURnTjf4nB+vTyXjvedk7/8dU3Eb8qCnwJnDyYLLf2WjvcDZ2NK6f2Q2SyHdOnJ6+ykB+nD69C2nIXgnFdLGTejLsDtR8eCDngpXjrcJvwKKz1UgdFgjkJR1TkQGcoQFfNZSS1kdjwKyqxYjRW8SEKwUJzmkAvDFfHNUuAHOfJQPPELzrOPrnowVUJDNzSu4mVTQtI6s3ZIPvrtxNqvgqIfE5SmLoRZFb+M67XP3bfZCmHuYjl0V3miyzQszEsfq3dwJuGt1/PzrzUDHFR6DiNx7wl3m6D8gyP+mCLVL1SXfWHa+1KLFHA54anaYKvR0fvKzGosIKFq9pXH4r3RTrBkZPERLgllqgJCaJ3AHTRGphpaev6mGVZ5RlAM5Lbf0pGj695yaL4a3/JTItGDqUY5LofS8Xo5VKxifaQVRIv4ffm9S/aQ713y9th88dgtYgVHAZJO+nGJwi+nt+058EnJrnfinHcB4Y8JSJsI0q6rDl72b/tnEIN55cqi7qBdsiKMTKVaLIYvsFItTYFAmK3YmankLUNTwWUURYDH6Qq/UkQDu8ioXADGzkcZtdQ8wKAWZTUvWlRMBEdZZsatBuM1Jfx4yfA2RYoGnqOHe4mDK8wUzIMgtC8yw6+FF6qBgm6zQa8r/vvX03Z7rvnfvHfDMd8v/qc+I6/7w5Rwcp1Kcbty8h6/0Owh+UW7QTrOkxDLZuieM3HNVpT8ayolb972w0yjFbiwrdMlxxfg9OvM8hL9ulQt49/dS7fpS2vdMiRxjhdlgLlCETcuK5e5eo2HmTJScJsDisI5/DcuEuNZYgZSxajItYWWrZTk6yEwHJbVkm1VzhkZTTIxBBWrgZLV8DBCqTmuxipKUZhVQllDR7OqjPXQ6JiSRnIugjIw3hEKHpKbuKQuE6w5rcPJUuqG+ddig2OZ1HrxbpyKuRs/Gquhbi3O3caFhn1QXOyEi/2EvzaMGo0Lx0+huqpSGAjeAN72P4KefUz03ISes4tqxGMHrv+f+mbZRsIzLJuPSB73FwiXS5/voD6FktKHKEGTvPGcCFLuho/IPjYcF0YxKChtBo0uJGiKtROI7kKOVIsxQoGl4oxQJvuQuNGyVJcCWUqYOhaSgW5LTFpXoLkyc5RhDMrMc7FkHFZjGatgLh0TnBcYFR450JKsSl3xqrRyWlfrQMJTBoVOlkD0qoHo1ZkntDWdS1+v/qWrlLAY0mHfn9tSbbvdxas89+4aVnmOA0LgF8+lB8mAgut2Q+uKZyi2vjWoWmq120kbdGePmLTBeSsZHk5mK2Ln+o/abY9UOn7X4wI+fUfhzMu1dRXvB/2pZL4MbL3PrYyaxmx5mvqJSggGkqVd9CVqT8lfcKJkKCpzyEjW3HvwInBpvHVhFWUgauJPIsD+OctAOUJcjSWr1hk/FeFuOQ4aVktKFl+JkFGwiiDB1oNxoDBF8UGepvTI4WSxFtREZpSmc+0V9ag4DBOP+tu9fLyFZUOZQYbGDnDD96ecjbJ77agPxvgDX5Qm3NdjShNvTYjoZJ6pMU5GuKjHQ5eomu9b13dxe5l540evZTtNvdu5d/cEicvHFKS2bjScXEczIDxNsYdah0zqpRIfq83K4Z30zOG00jR3lDRvS0GPpqS5nbCYd/AYnPihDeQ3UihfqqNqipg00pcy5mjEh5wueB8jJHQyU779cf8W0svme4ArA4zSK7y6eRVwWzUqhzHiBYq9vblaeHupmYZ7xKVm3Jed8LDXereBMM/jCT3pkgGj5eoh4THl8UxgbcNQFrQSmbsQ2ceESDd7TSefdP0bw+U3ZlELmg/YfN4H0FGvOjiUEHbjINoTMroZbeGfMMFb7xMG9QIKbRz8/GJtCR6PrlilHlvRaAPPFI8G/WF00rwKG0PC8weRTA1yNZagxWpRHFcGQEHBJBOAHmaptpxa3jmALhlxayq6EIRMqxgYThbLB/2xLbSmXw83MDnsd8fnshu81JY2+L7vKWsQ5CYPVaxf6Q1IIz9mb0Dyeho27trLOG3f9rtw+NhQhk8BEXnZf79o374b/ghFswvwKYbVpskwL6TlLiTuMSGRE8ejk6vEljFFud9TyXPX96dCzXv7VpXz8fZWSO9clPHw2dBatCUMbqlKc/JUqf3q8I3VIu3tanq6ncespie4G2LcXCN2G6t57fjotJe/nbpvYUt9A8k16/69TF7sp79wK0rwG7C/jeypp+qy1nr5t7/R2tPQ9D+9ng634wv3Xhs3j/raWqMavuxUk16onO/XQ/jYnHYZVvZXKL0ZzajgW9eqcQBlCJG+Qx0cvqoDpnY1j6cDpJygWSQejH2PbsJyKR9b4rFJ4SXcaJB1Fxb5oLCoSTWqhfs7FJzDGnRg4lQn9FLe9ofDC2HV0lTuNEaf9QE9F3kuf+y3ivMqiIPg0miNqQIXZIJRrYvagxO0EgdHx0VeS+WYWTywSguuWitZPfc6CIqj1wI2omoCeXdgMZ7VqPxqLJUrnMdCnmI1RnllnVMgJkFkuH7gcSEIqTC3uSlZ0pItvjypS2OWxf/DpCNTuxlxOMWhqXyL+Z1RO4Tc7Z8PoXZTC4DL2vYY4WtLFtvBKxgs9DNcqO3gGy0K+hjRyJAVd/B5U5zQM1d6i49MN3FKCKIMY5c13BdhXEyAxispKuPBQULByBS4LpxqCBowoJanUoPPAfLVaKHCfS7GFFVSLLkGFxJzpnpJnmcFplkLwaIx8ErhMRJ+btJbTlxPzhG90/Mz7SzwbYwu7w+UrZxhQRYO5LgK/AmSLcqMr5ZXaks79q/9GnhOPCh4qrX0HU7mSp/63IAK/zqrLz6ru8PpfbFWNl18WRQraUjzXczUBbNJwlID0Ci81dK1JVW6lRsmMvlQPFSZySzxaDKH9XMGHnXUysMhh55TWsCOcp+h36ACYUUBApz0Ikriq9FKKUJxmXNkMkSBCV/hwIdCjTeZCI66s1uuYX4l9KvmhBZvCo9OvopWt6tgCp+idclU/E3ldg8zrh8WMzthZufdlc74YELBE/1oval2lKbWX7WLnuWrzGzqfGqSBzYEOW4GN+7YA8XtQ9oD1YrYhpO5kJ1b5QyO+5/u9+tvzXFW/WqN4SlQXYhWNko/4dBK2DxcPivGJgjKFXpAOOvRJgXyGHQOkXOHdw5qqUrQ3FCzXx4r93j72tuqIStCJQbOsRoLcuEZtdBJRhQfUs3goY5HSdFDG3jhXrFYYjUsenAVwqGlsgwnV+xivlz4viunt/1iwXmW73b5+HXlWeUsYynaWtBp4c1ZMCS72sZhIlxgy5EZZzwUrZJOaed9ztpACRtF8WNhROA6+iiq5r4YbTApnJDGVidXo6miVJGFh8JyDBFkzEqXUxKh8qCt9CB1mQKJEEUNuRwg4RMkfFV7cg+IPwYBf1QIWp3AcDJfOTh3eVgUZrXWDp90MLgDgZIPC4GaIFB/v/q8uzCoR4VhinAqzT/JJ0hhm7rapXWlzVXywGvY9m2roK81V6VhH7G5Ku5LTLjc6Djb4JjRkXXH2da5+I9AIh4TEjNBYsTkYm0+9ieia5tciI6RSwoWUijkM8WT9il174PFhfK87Kq0pdYM1LHuvEp5xvLPVIRDy07E6rbhveunzjt9GIqLKUmg5UGNTK87nPDDcCjeL/3sLmucbagvthunx3vEduO4r0lpm2V1198T/XPw4Ar5JeR9OJ7GgB91RqKw3uv+7/sS5GO+BD/py0VZ90jDIwX5juF13Gej/LraCGDGvd+bMIeOwqL4OzznPpAjE3pqOQU2evoeDn34qpHxj0o39JT2oqdW+TPNUbenLl+Fr7enWfrLlTJoZiaF7jDuonSp87zZ/jTD87IMTgTPUZcpdZRuYyTrTF+AclJSx1Ir4UKoGEuC1JSkdPVMWVty4TLkpKUOMcmQZLGFKUpGiEFYqilbDga/UHjq66g1PCKBT4Wr3EkKLEuoYZkEvipYwF9iHvbW0q1gfjhZFHB+DtOwwcR/GBbxoLDwCRYuLh2nrup13rt8IoW2P2z6UxzPX4aNBRbJQVd8Zbri8v1hn6TpF1YFXrd1Fe7qIXWVnvKD6GRRSH0TsKGtIG3JOTRdurd52W2sNr+cnsOXAZMPCpiaAPusOOwSh79ZGzYB9UW6h7Eeku7pKUynF2G6sfphwmC2OLuQk5Gxdc+7gf5+HQ79mHCoyYYpvlxgGG3/sMIw7GbyW5G2pn3atQvlrLRllL7kA8ZmOUy4tG4NlGQ2JZpxPKEAJs6EUKnUKdvMWOaYMzXij1pKWu5UtThjpCja4PmFTK74IKpwq7GiVIpCbgRVyHh0k70NmK6miBhdTFzbXKHyk6iS1cG2T0l3Wi1N1jUWJB2/84nFYz6xbaFFOrku8ZgJwlXm6lIgrusmz9mq15nOX0TJugdFyU8oLVsn34Zmkbn7x7DwD4mFmWidYctkzPewqd327VYcmv6++T9/XpcL1HGzNstoSSpOEWhztTEEbmLs6l90BItSEg8I9qV85DKLikeCdfQxKzyXxnNaoZQCD1NCOY3f1cBJpRpWo/HiE094dM9lFDyD/MiauIF7AQQDNVqyDkacgwg6WyiLzrR62uHkupEa2ZJCO/ik/ds7zoe+x7RkcbWv5RBc+WkMsYy6dNtRT8fby4fGOZBU2vQOjk1xE7s1Yrb2avloWaKHaFgbMiUQ1mq1SdQwQYlsXKwiwAgwlYRmvnhVvC5VaBOiZTJKcFe5GszrUCyDQfIiWfxqih5s2cMl8lnwAlbrjAsyJuOl83WEx03wXDfc23bfymGC6Pmw/07ohMP+tMv3QBrya+BxUgvxeHoZa09e4Y8ffidW7kGxmqK2Rl0xtA2QOlJOfzkXI15ic630kEpwXsMQwDiuK0lu789I4zzi/oxUgTgBsd59bwJj+4XdmCdUvoyHfkw89CQYet5gjXbC/dZdwl6zQoDcba96ybYr5+UAl61h15lpeCrpIeKK6o/FeR0Qw6+LeCvtCSdCBB9VGZw12WTIENUcmIseBkgbeCRWSW6ztEFRyhquicyQ72JXg4liMDuKg23CtfC2gYyTILtJalaT0VQEm43UmIlRFT0ANAmM1telR4DkUGgz4YbA/y9g9IMCY9oKKZ3caBQ/TaEhQZL+cN6fL/b3p1ULkpIdH0Kln+19fifJEffzoEmOcNIbZHbWpm6GzaWcklC6oZLvIDMlWs02frylq+8BZvnDAuYnwPzal+5Ph8Gof0J+rgoAx0KLzw37PfXdGPLDqe+pnMBclRM0LC78kHbA+Dgjd5l+w1Wz6q3NfluW0H0VIveQOzLzqZnqcLLu4wbXqe4PV4q7NXKjmTbfSXW4br0aeQeOcz/Vh4NDTnAsV8Mo5jRktZwhGTG62/BvAGoIVw0Zd19GRj4mMq04cTiZ+xOhO3wPH0MNV38pdJvWFGLoZ6uAUyEXFDWlXW+GipSp+m2243kO/Y3U9NubetMNPeSm3rgxN0HmljnYU9v4GXo3CuRiNy2pXuM4S8WeCuWW0M7qfQYZfJsaaQ2t5Qnfy69/HWn3oEhPecZ0sqhLFZvvhYpPSCZXuTfXBXWtyiTu4/64+av4ammqdY9ZmmqnvhR0cnsDnqFG5+amRF+WCfeos2+ql6OTJhMgkxALzIbvh/2UzXBz06EpL+RWovRXcXnYueInXPxnuJT/GCwyRJYV6LDEzVRmz7CMVNFb8+TbVlWJa0F1MQycWKsc4bopWChYNY7JUinXvoaaAkgD00nB7knBFGQd9JmqSFdjueisLDCSJfqM6zhVwlvhSlZCwtxpRX3chA0SY+khFdr6RhXp5Dp9czebLR9fmyv3Ht6zh3z48cmvo4SnS75m21h33j5k1iT8s610KToMTuG1UtSfemqTzthl1Uk3HLyrGa5UrHBuhAyS4QF58kVU6lVN28ZyRcFBOEDKVJiiYiDrvhB+Rda4GsuYlKLiFrrQM++V9FJlTIeigWdI2VjcBlVcRPhZPIcffxjnhb9VRdZmxKzJwXWi92xJbnKlhhrElvV9/t4NW6uHPP9Em6c4Jlv1PGHkZyuWI0bWFZUTUyJylWzyFe4i3ErHwQ81A/8ThvogSUDAbBVRUFAUDqUvWUKW5GqsYFXJ2cnkRK5JxmCN0obkz2WlLWW9U6p8sLXQzqv2xx9aE0dhV4Xrb5QSsjat/+tf/3m1+4/hSbCaC3fCQJCnJxbzvaWEGg0pVCe3YpD/XAUXHjZRwvLFYoNWvlINDVxnr0QqOuItU/66yToYFkKUZjVaAiEWIjlaaKF9ia1g3pQcNAekgAAD+5BY0K5KCBkMaYukL1L+24I1CBR14/ukX9+5uc754rW+MEHbaCHrtMah9RkRd6M2Cbaf1UjLQyxC5EMptGsfTfkIWSgFrgB1QbGYDazqIErWMAVja4+Ywno0J130gJF5Eak2vzrvJHWwyzEkU2v2mElWqhTgYyRdgcg4VYT/HBGaNtRNf2iEnT7up7W+ddvutRsa57em2ef8wG9QPOdG+l+BzD8mZG2JYbbAAOo/tdA5e0vkUF4E6URJD5dSpIG4D5XNQ7/wZWmlzo5xQ7VkKnh/BmRaapiV0oBvMJOhFmsi6ugg8clp6lFYnOTeFxOopxopGctjqQY6A/MmGmULtX1YjSU9KIeq1sAShZwKfh/WpgYmeBFK0myyujCrWBAhWOjattCwXmboDldgXNnbEY8/CIN+RBhM21jsOhem7eHXtvCbrzmNWxte4i7DFTdmB4xEgbGHFs8+czVtMcGMvlrNHfdoJ40HP79ymADcJwAoAc6G1qkqS4lO1BY1GGNzCNJKmjDOOS8q9C+MbVyNBgtdKiXv2Uj7+CkeRB0qkuh7oCeVZ9qb1GQfWWWVdgczbeunCw5/eOfHr4NhHhMM3pqJ86tm4n90Yf/O03OhHvTpp4R0I+crStOcWLb7a1PiEsC+k9yRoOUFnCaG5wLnmXoE85YRbqx8cq3rWBm6+lgB3ZCoD6512sOn0tQpF3goRRWG4J5Q+8aUSBvwApYIllWcANFajWWUt3DYS8C/E3P4ILlaRaE2W9A5UYD9U0hPBOZdCmoIRE5xyKt4xvthnwqs7Cx0MzHyw74fVpO6C+G4H+CpFixYJO6p/xnPU+dsDLfSksYIzjScVg03Unq8WVOcovJoXy0Ydq3MwDPxVSaYy6RgXRWYFyRIV5AssRrLK2hDaWryHD6uMxx6WGgYTl6ZY1C1iTZ21jJrmZKqhtBoE8IuEyjJKfnpXlv9+U6LqaOO8+usF9Beiyek1UHPsj7jME4OqfiTanWXUuNGB9eaQaP7FFjigvb7KlWKOGzHo122YOgRfooJVoNtWHBrnTWtCq3GUglf98boKKgo3Emm4ecFTs3JLXOxKEhnAWqBS0giOWqurba6a3sx7i290o/DHtKboWvv8vVnGUqMAi4VULbTrqT43dEvM/4Js3v0T2kHV4XH1IY6qAcoDgGzSDWlmf4zAe4GN6DgiasIYpU5dw7/L4YpHvlqLLxcxwRcl+x4SsFbO9Th4oFpLkA5YFKUZF3QoF7WU6aTb9E9LxZhrFxub7951XTmBO/1MF5Qu7LNN6Ooz0MbkLYl53HY2Hy+F2eLSw8bw87S4G82XnWOwjqY0konB7GYwPVXHVRGYhoStKyFwgW8oWZKQg4O0NM2AoKxgEknaFtOCV8Y7IM7Yy3gZ7iYC13qajAB963ATxRJZKZqAP+E2kpVwmlmmKqugMEahRekQVhSInRblPB6a2EQ98P7aShK2+770xsw/bbffpuRkeUFA0+NN9pKRaczDIvVYMbKtb7fBMjM39O2FQYkl2kbUTxMgF2hfJ/KQ4SasJr6O0bDTVYpV0/JdrkwCYAEtegXwYHerAaDiCb4UbSXtYKC1yw64W0WxmAYFVlS0UIiPXwBSvKlLoe+Bcj8MsVwLNed2p0s93VdfbjwenkK0SnYFanBJMMZhRYQExpmWIxOL4PvDk+32FRp6biapHl0hbOoYJOtsphUtKc7rJNQlLPAtfFZwC1RAkqFrwbzmGzK+ZyBGTUgN4raXvpkufWUuqclJcgrKCOoLacw6QQfBZeOy0ygc2f2Ns8uaVL9co/7dfrCMm/q0326S8F7BheBbQAUUzo3taychwrc2Kc9sRKV4TZy7ywrlUkJXQK2DTfMquSDgXqGS69gdnKoUN/ZEyvXDPrV+rgajUdhoPElkCqYqaqQoPgKm8eql1Uyi2lo4Tv4KhJABWotKZGO16gNVRCzErrrbYEWSe65u7kV0J1QkZg2/HiwUFFLnhJaXYUW0zZ8GxvIxsP+dSYvTTLG3jev4XaWBk1yipRKR7oxTSFF0TKkoCeevPWtUwIeWkgfQ9QKBojDYlGyBtisMi7GDGPuuBPOJvCXgseKxGKTSfD/VVCrsZSVYDQmFvwBWpmBAmUHbhMdU5AEVSOYHwwh0IU91B4YtJgnHdd1fVNc+VYY8fLZQptCUKUj/qxhRcNZmwoj1g6ucElYFwWFvHQidSdZoLiwhmjAIDvlJY+xgmpYZiKsgtWOAq6FtgYUq7GSKcOKM6g6+FqB8yuHNrlwEYSk1gQqB4iLZpCYajP78Yf/BxzgFts="

manifest_search_paths = [
    DATA_DIR / "decisive_validation_manifest.csv",
    DATA_DIR / "results" / "decisive_validation_manifest.csv",
    Path("/kaggle/input/indorad-vqa-research/main/results/decisive_validation_manifest.csv"),
    Path("/kaggle/input/indorad-vqa-research/results/decisive_validation_manifest.csv"),
    Path("./main/results/decisive_validation_manifest.csv"),
    Path("./results/decisive_validation_manifest.csv")
]

manifest_path = None
df_eval = None

for p in manifest_search_paths:
    if p.exists():
        raw_b = p.read_bytes()
        if hashlib.sha256(raw_b).hexdigest() == AUTHORITATIVE_MANIFEST_SHA256:
            manifest_path = p
            df_eval = pd.read_csv(io.BytesIO(raw_b))
            print(f"✅ Loaded authoritative manifest from disk: {p}")
            break
        else:
            print(f"⚠️ Notice: Found {p} but SHA256 did not match authoritative hash. Skipping.")

# If not found on disk with matching hash, unpack bit-identical embedded store
if df_eval is None:
    print("📦 Restoring authoritative manifest from embedded bit-identical store...")
    raw_b = zlib.decompress(base64.b64decode(EMBEDDED_MANIFEST_B64.encode("ascii")))
    assert hashlib.sha256(raw_b).hexdigest() == AUTHORITATIVE_MANIFEST_SHA256, "Embedded manifest corrupted!"
    df_eval = pd.read_csv(io.BytesIO(raw_b))
    manifest_path = DATA_DIR / "decisive_validation_manifest.csv"
    manifest_path.write_bytes(raw_b)
    print(f"✅ Successfully materialized authoritative manifest to: {manifest_path}")

# Strict Invariants Validation (Zero Compromise)
assert len(df_eval) == 150, f"Manifest length must be 150, got {len(df_eval)}"
split_counts = df_eval['split'].value_counts().to_dict()
assert split_counts == {'test': 92, 'dev': 58}, f"Split counts must be {'test': 92, 'dev': 58}, got {split_counts}"

test_df = df_eval[df_eval['split'] == 'test']
dev_df = df_eval[df_eval['split'] == 'dev']
test_clusters = set(test_df['canonical_image_id'])
dev_clusters = set(dev_df['canonical_image_id'])

assert len(test_clusters) == 35, f"Test unique clusters must be 35, got {len(test_clusters)}"
assert len(dev_clusters) == 23, f"Dev unique clusters must be 23, got {len(dev_clusters)}"

leakage = test_clusters.intersection(dev_clusters)
assert len(leakage) == 0, f"Fatal cross-split leakage detected: {leakage}"

print("\n" + "=" * 70)
print("📊 AUTHORITATIVE FROZEN MANIFEST AUDIT REPORT (REV02)")
print("=" * 70)
print(f"  - Total QA Pairs            : {len(df_eval)}")
print(f"  - Total Unique Image Clusters: {len(test_clusters) + len(dev_clusters)}")
print(f"  - Primary Test-Only Subset   : {len(test_df)} QA pairs across {len(test_clusters)} image clusters")
print(f"  - Development Subset         : {len(dev_df)} QA pairs across {len(dev_clusters)} image clusters")
print(f"  - Cross-Split Image Leakage  : 0 clusters (OVERLAP ZERO VERIFIED)")
print(f"  - Manifest SHA256 Verification: PASS ({AUTHORITATIVE_MANIFEST_SHA256})")
print("=" * 70)

display(df_eval[['qa_id', 'canonical_image_id', 'image_id', 'split', 'translation_status', 'question_en', 'question_id', 'reference_answer']].head(5))

📦 Restoring authoritative manifest from embedded bit-identical store...
✅ Successfully materialized authoritative manifest to: /kaggle/working/indorad_validation_rev02/data/decisive_validation_manifest.csv

📊 AUTHORITATIVE FROZEN MANIFEST AUDIT REPORT (REV02)
  - Total QA Pairs            : 150
  - Total Unique Image Clusters: 58
  - Primary Test-Only Subset   : 92 QA pairs across 35 image clusters
  - Development Subset         : 58 QA pairs across 23 image clusters
  - Cross-Split Image Leakage  : 0 clusters (OVERLAP ZERO VERIFIED)
  - Manifest SHA256 Verification: PASS (988f57a6512223337bb5ab3b192f1802e38bfec2f6b262f51607381faa74bf52)


,qa_id,canonical_image_id,image_id,split,translation_status,question_en,question_id,reference_answer
0,47,6185dbde43bcbc0c,img_47,dev,CLEAN,are structures associated with the midbrain lo...,Apakah ada struktur yang terkait dengan midbra...,YES
1,58,6185dbde43bcbc0c,img_58,dev,CLEAN,is the midbrain identified in this section?,Apakah bagian tengah otak (midbrain) teridenti...,YES
2,72,376a2117295e209a,img_72,dev,CLEAN,are the margins of the cardiac silhouette normal?,Apakah batas-batas siluet jantung normal?,NO
3,73,376a2117295e209a,img_73,dev,CLEAN,is the heart border normal?,Apakah batas jantung normal?,NO
4,74,b973d542fc5ecfd3,img_74,test,CLEAN,is this an axial plane,Apakah ini adalah bidang aksial?,YES


In [5]:
# ==============================================================================
# CELL 5: MULTI-STRATEGY IMAGE RESOLVER & CRYPTOGRAPHIC VERIFICATION
# ==============================================================================
import io
from PIL import Image

search_image_dirs = [
    DATA_DIR / "images",
    DATA_DIR / "data" / "images",
    IMAGES_DIR,
    Path("./main/data/images"),
    Path("./data/images")
]

# Dynamically scan /kaggle/input for any directories containing png files
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").rglob("*.png"):
        if p.parent not in search_image_dirs:
            search_image_dirs.append(p.parent)

def resolve_single_image(cid: str, img_id: str, qa_id: int, image_name: str, 
                         expected_pixel_hash: str, expected_w: int, expected_h: int) -> Path:
    candidates = [
        f"{cid}.png",
        f"{img_id}.png",
        f"{img_id}",
        f"img_{qa_id}.png",
        f"img_{qa_id:05d}.png"
    ]
    if image_name:
        candidates.append(str(image_name))
        
    for sdir in search_image_dirs:
        if not sdir.exists():
            continue
        for cand in candidates:
            cand_p = sdir / cand
            if cand_p.is_file():
                try:
                    with Image.open(cand_p) as im:
                        if (im.width, im.height) != (expected_w, expected_h):
                            continue
                        rgb_arr = np.array(im.convert("RGB"))
                        calc_hash = hashlib.sha256(rgb_arr.tobytes()).hexdigest()
                        if calc_hash == expected_pixel_hash:
                            return cand_p
                except Exception:
                    continue
    raise FileNotFoundError(
        f"Could not establish cryptographic identity for QA ID {qa_id} (cid={cid}, expected_hash={expected_pixel_hash})"
    )

resolved_map = {}
missing_images = []

for idx, row in df_eval.iterrows():
    qid = row['qa_id']
    cid = str(row['canonical_image_id'])
    img_id = str(row.get('image_id', cid))
    iname = str(row.get('image_name', ''))
    exp_hash = str(row['pixel_sha256'])
    exp_w = int(row['image_width'])
    exp_h = int(row['image_height'])
    try:
        resolved_p = resolve_single_image(cid, img_id, qid, iname, exp_hash, exp_w, exp_h)
        resolved_map[qid] = resolved_p
    except FileNotFoundError:
        missing_images.append((qid, cid, img_id, iname, exp_hash))

# Automatic Self-Healing Materialization: If any images are missing from disk,
# fetch the exact QA rows from the authoritative Hugging Face repository (Lab-IS/IndoRad-VQA)
# and verify pixel hash cryptographically before saving to disk.
if missing_images:
    print(f"⚠️ Notice: {len(missing_images)} / {len(df_eval)} images not found in local disk paths.")
    print("🌐 Materializing images directly from authoritative Hugging Face repository (Lab-IS/IndoRad-VQA)...")
    try:
        from huggingface_hub import hf_hub_download
        
        p_train = hf_hub_download(repo_id="Lab-IS/IndoRad-VQA", filename="data/train-00000-of-00001.parquet", repo_type="dataset")
        p_test = hf_hub_download(repo_id="Lab-IS/IndoRad-VQA", filename="data/test-00000-of-00001.parquet", repo_type="dataset")
        
        df_tr = pd.read_parquet(p_train)
        df_te = pd.read_parquet(p_test)
        df_all_hf = pd.concat([df_tr, df_te], ignore_index=True)
        
        recovered_count = 0
        for qid, cid, img_id, iname, exp_hash in missing_images:
            hf_row = df_all_hf[df_all_hf["qid"] == qid]
            if len(hf_row) > 0:
                raw_bytes = hf_row.iloc[0]["image"]["bytes"]
                im = Image.open(io.BytesIO(raw_bytes))
                rgb_arr = np.array(im.convert("RGB"))
                calc_hash = hashlib.sha256(rgb_arr.tobytes()).hexdigest()
                if calc_hash == exp_hash:
                    save_path = IMAGES_DIR / f"{cid}.png"
                    im.save(save_path)
                    resolved_map[qid] = save_path
                    recovered_count += 1
                else:
                    print(f"❌ Hash mismatch for QID {qid} from HF: expected {exp_hash[:16]}, got {calc_hash[:16]}")
                    
        print(f"✅ Successfully materialized and verified {recovered_count} images from Hugging Face.")
        missing_images = [m for m in missing_images if m[0] not in resolved_map]
    except Exception as e:
        print(f"❌ Error during Hugging Face image materialization: {e}")

if missing_images:
    print(f"❌ FATAL ERROR: {len(missing_images)} / {len(df_eval)} images failed cryptographic identity verification!")
    for m in missing_images[:5]:
        print(f"   Failed QA ID {m[0]}: cid={m[1]}, expected_hash={m[4]}")
    raise RuntimeError(f"{len(missing_images)} images failed cryptographic verification. Stopping execution.")

print(f"✅ Cryptographic Image Identity Verified: All {len(resolved_map)} / {len(df_eval)} images match pixel SHA256 and dimensions perfectly.")

⚠️ Notice: 150 / 150 images not found in local disk paths.
🌐 Materializing images directly from authoritative Hugging Face repository (Lab-IS/IndoRad-VQA)...


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 59.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.2MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

✅ Successfully materialized and verified 150 images from Hugging Face.
✅ Cryptographic Image Identity Verified: All 150 / 150 images match pixel SHA256 and dimensions perfectly.


In [6]:
# ==============================================================================
# CELL 6: CORE EVALUATION & STATISTICAL FUNCTIONS
# ==============================================================================
import torch
import torch.nn.functional as F

def extract_target_loglikelihood(model, processor, image_path, prompt_text, candidate_strings, target_device):
    """
    Evaluates conditional log-likelihood of exact candidate target strings
    (e.g. 'Yes' vs 'No' or 'Ya' vs 'Tidak') given the multimodal context (Image + Prompt).
    Avoids single first-token logit hacking and includes prefix-match assertion.
    """
    from qwen_vl_utils import process_vision_info
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": str(image_path)},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    
    prompt_str = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    prompt_inputs = processor(
        text=[prompt_str],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    prompt_len = prompt_inputs.input_ids.shape[1]
    
    cand_logprobs = []
    
    for cand in candidate_strings:
        full_text = prompt_str + cand
        inputs = processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(target_device)
        
        # Verify prefix match to guarantee zero boundary slippage
        assert torch.equal(inputs.input_ids[0, :prompt_len].cpu(), prompt_inputs.input_ids[0].cpu()), \
            f"Tokenizer boundary prefix mismatch! prompt_len={prompt_len}, inputs_len={inputs.input_ids.shape[1]}"
            
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0]
            
        target_ids = inputs.input_ids[0, prompt_len:].tolist()
        if len(target_ids) == 0:
            cand_logprobs.append(-999.0)
            del inputs, outputs
            continue
            
        target_logits = logits[prompt_len - 1 : prompt_len - 1 + len(target_ids)]
        log_probs = F.log_softmax(target_logits, dim=-1)
        seq_logprob = sum(log_probs[i, target_ids[i]].item() for i in range(len(target_ids)))
        cand_logprobs.append(seq_logprob)
        del inputs, outputs
        
    return cand_logprobs


def score_binary_candidates(model, processor, image_path, prompt_text, candidate_strings, target_device):
    """
    Runs sequence scoring and returns normalized binary probabilities.
    candidate_strings: [affirmatory, negative], e.g. ["Yes", "No"] or ["Ya", "Tidak"]
    Returns: (pred_label, confidence, p_aff, p_neg)
    """
    logprobs = extract_target_loglikelihood(model, processor, image_path, prompt_text, candidate_strings, target_device)
    max_lp = max(logprobs)
    exp_lps = [np.exp(lp - max_lp) for lp in logprobs]
    total_mass = sum(exp_lps)
    p_aff = float(exp_lps[0] / total_mass)
    p_neg = float(1.0 - p_aff)
    
    if p_aff >= 0.5:
        pred_label = "YES"
        confidence = p_aff
    else:
        pred_label = "NO"
        confidence = p_neg
        
    return pred_label, confidence, p_aff, p_neg


def compute_discrete_risk_coverage(scores: np.ndarray, correctness: np.ndarray):
    """
    Sample-level discrete selective prediction evaluation (Geifman & El-Yaniv, NeurIPS 2017).
    Ranks samples by reliability score descending: S_(1) >= S_(2) >= ... >= S_(N).
    Evaluates empirical selective risk r_k = 1 - (1/k) sum_{j=1}^k Correct_(j) for k=1..N.
    """
    scores = np.asarray(scores)
    correctness = np.asarray(correctness, dtype=float)
    n = len(scores)
    
    order = np.argsort(-scores, kind="mergesort")
    sorted_corr = correctness[order]
    
    k = np.arange(1, n + 1)
    cov = k / n
    cum_correct = np.cumsum(sorted_corr)
    acc = cum_correct / k
    risk = 1.0 - acc
    
    aurc = float(np.mean(risk))
    
    cov_trap = np.concatenate([[0.0], cov])
    risk_trap = np.concatenate([[0.0], risk])
    trap_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
    aurc_trap = float(trap_fn(risk_trap, cov_trap))
    
    return cov, risk, aurc, aurc_trap


def get_risk_at_coverage(cov_arr, risk_arr, target_cov: float):
    """Find selective risk at closest discrete coverage level."""
    idx = (np.abs(cov_arr - target_cov)).argmin()
    return float(risk_arr[idx]), int(idx + 1)


def run_image_cluster_bootstrap(df: pd.DataFrame, score_cols: list, corr_col: str = "correct_id", 
                                cluster_col: str = "canonical_image_id", n_boot: int = 1000, seed: int = 42):
    """
    Paired Image-Cluster Bootstrap to account for intra-cluster correlation.
    Resamples unique physical images with replacement.
    """
    np.random.seed(seed)
    unique_clusters = df[cluster_col].unique()
    n_clusters = len(unique_clusters)
    
    cluster_dfs = {c: df[df[cluster_col] == c] for c in unique_clusters}
    boot_aurcs = {col: [] for col in score_cols}
    boot_diffs = {col: [] for col in score_cols if col != "score_m1"}
    
    for _ in range(n_boot):
        sampled_clusters = np.random.choice(unique_clusters, size=n_clusters, replace=True)
        boot_df = pd.concat([cluster_dfs[c] for c in sampled_clusters], ignore_index=True)
        corr = boot_df[corr_col].values
        aurc_m1 = None
        
        for col in score_cols:
            _, _, aurc, _ = compute_discrete_risk_coverage(boot_df[col].values, corr)
            boot_aurcs[col].append(aurc)
            if col == "score_m1":
                aurc_m1 = aurc
                
        for col in boot_diffs.keys():
            boot_diffs[col].append(boot_aurcs[col][-1] - aurc_m1)
            
    summary = {}
    for col in score_cols:
        vals = np.array(boot_aurcs[col])
        summary[col] = {
            "mean": float(np.mean(vals)),
            "std": float(np.std(vals)),
            "ci_95_lower": float(np.percentile(vals, 2.5)),
            "ci_95_upper": float(np.percentile(vals, 97.5))
        }
        
    diff_summary = {}
    for col in boot_diffs.keys():
        diff_vals = np.array(boot_diffs[col])
        p_val = 2.0 * min(np.mean(diff_vals <= 0), np.mean(diff_vals >= 0))
        p_val = min(1.0, max(1.0 / n_boot, p_val))
        diff_summary[col] = {
            "mean_diff": float(np.mean(diff_vals)),
            "ci_95_lower": float(np.percentile(diff_vals, 2.5)),
            "ci_95_upper": float(np.percentile(diff_vals, 97.5)),
            "p_value": float(p_val)
        }
        
    return summary, diff_summary

print("✅ Core Evaluation & Bootstrap Functions defined.")

✅ Core Evaluation & Bootstrap Functions defined.


In [7]:
# ==============================================================================
# CELL 7: RISK-COVERAGE PLOTTING ENGINE
# ==============================================================================
import matplotlib.pyplot as plt

def plot_risk_coverage_curves(df: pd.DataFrame, score_cols: list, save_path: Path, model_title: str):
    """
    Generates publication-ready Risk-Coverage curves for all 5 estimators.
    """
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
    
    color_map = {
        "score_m1": ("#1f77b4", "M1: Confidence-Only (p_orig)", "-"),
        "score_m2": ("#ff7f0e", "M2: Visual-Only max(0, Delta)", "--"),
        "score_m_dir": ("#2ca02c", "M_dir: p_orig * max(0, Delta)", "-"),
        "score_m_norm": ("#9467bd", "M_norm: p_orig * (1 + Delta)/2", "-."),
        "score_m3_old": ("#d62728", "M3_old: Historical Unsigned", ":")
    }
    
    correctness = df["correct_id"].values
    n_samples = len(df)
    n_clust = df["canonical_image_id"].nunique()
    
    for col in score_cols:
        cov, risk, aurc, _ = compute_discrete_risk_coverage(df[col].values, correctness)
        color, label, linestyle = color_map.get(col, ("black", col, "-"))
        ax.plot(cov, risk, label=f"{label} (AURC={aurc:.4f})", color=color, linestyle=linestyle, linewidth=2.0)
        
    base_error = 1.0 - np.mean(correctness)
    ax.axhline(base_error, color="gray", linestyle="--", alpha=0.6, label=f"Random Selection (Risk={base_error:.4f})")
    ax.axvline(0.8, color="red", linestyle=":", alpha=0.5, label="Operational Reference (Coverage=0.80)")
    
    ax.set_title(f"Selective Risk vs. Coverage under Indonesian Language Shift\n[{model_title}] (N={n_samples}, {n_clust} Image Clusters)", fontsize=12, fontweight="bold", pad=12)
    ax.set_xlabel("Coverage (Fraction of Questions Answered)", fontsize=11, labelpad=8)
    ax.set_ylabel("Selective Risk (Error Rate among Answered Questions)", fontsize=11, labelpad=8)
    ax.set_xlim(0.1, 1.02)
    ax.set_ylim(-0.02, max(0.45, base_error + 0.1))
    ax.legend(loc="upper left", frameon=True, fontsize=9.5, facecolor="white", edgecolor="#ccc")
    ax.grid(True, linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"📊 Saved Risk-Coverage curve: {save_path}")

print("✅ Plotting Engine Ready.")

✅ Plotting Engine Ready.


In [8]:
# ==============================================================================
# CELL 8: PIPELINE EXECUTOR WITH STRICT REV02 RESUME
# ==============================================================================
import gc
import time
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm

PROMPT_EN_TEMPLATE = "Answer only YES or NO.\nQuestion: {q}"
PROMPT_ID_TEMPLATE = "Jawab pertanyaan radiologi berdasarkan gambar berikut. Jawab hanya dengan YA atau TIDAK.\nPertanyaan: {q}"
CANDIDATES_EN = ["Yes", "No"]
CANDIDATES_ID = ["Ya", "Tidak"]

def execute_model_evaluation(model_name: str, df_input: pd.DataFrame, image_map: dict):
    safe_name = model_name.replace("/", "_").replace("-", "_")
    model_out_dir = RESULTS_DIR / safe_name
    model_out_dir.mkdir(parents=True, exist_ok=True)
    
    checkpoint_csv = model_out_dir / f"live_validation_checkpoint_{safe_name}.csv"
    expected_qa_ids = set(df_input["qa_id"].tolist())
    
    completed_qa_ids = set()
    results = []
    
    # Strict Checkpoint Validation & Resume Logic
    if checkpoint_csv.exists():
        try:
            df_done = pd.read_csv(checkpoint_csv)
            is_model_match = ("model_id" in df_done.columns and (df_done["model_id"] == model_name).all())
            is_method_match = ("scoring_method" in df_done.columns and (df_done["scoring_method"] == "target_candidate_sequence_scoring").all())
            is_rev_match = ("revision_id" in df_done.columns and (df_done["revision_id"] == REVISION_ID).all())
            is_hash_match = ("manifest_sha256" in df_done.columns and (df_done["manifest_sha256"] == AUTHORITATIVE_MANIFEST_SHA256).all())
            required_cols = {"qa_id", "canonical_image_id", "pred_en", "pred_id", "conf_id", "delta_p_dir", "score_m_dir", "reference_answer", "split"}
            has_required_cols = required_cols.issubset(set(df_done.columns))
            qa_subset_match = set(df_done["qa_id"]).issubset(expected_qa_ids) if "qa_id" in df_done.columns else False
            row_count_valid = len(df_done) <= len(df_input)
            
            numeric_cols = ["p_en_yes", "p_en_no", "p_orig_ya", "p_orig_tidak", "p_blank_ya", "p_blank_tidak", "delta_p_dir"]
            has_finite_scores = all(
                c in df_done.columns and df_done[c].notna().all() and np.isfinite(df_done[c]).all()
                for c in numeric_cols
            )
            
            if (is_model_match and is_method_match and is_rev_match and is_hash_match 
                and has_required_cols and qa_subset_match and row_count_valid and has_finite_scores):
                completed_qa_ids = set(df_done["qa_id"].tolist())
                results = df_done.to_dict(orient="records")
                print(f"🔄 Resuming from verified {REVISION_ID} live checkpoint: {len(completed_qa_ids)} / {len(df_input)} samples already evaluated.")
            else:
                reasons = []
                if not is_model_match: reasons.append("model mismatch")
                if not is_method_match: reasons.append("scoring method mismatch")
                if not is_rev_match: reasons.append("revision mismatch")
                if not is_hash_match: reasons.append("manifest hash mismatch")
                if not has_required_cols: reasons.append("missing required columns")
                if not qa_subset_match: reasons.append("QA IDs not a valid subset")
                if not row_count_valid: reasons.append(f"row count {len(df_done)} exceeds manifest {len(df_input)}")
                if not has_finite_scores: reasons.append("non-finite/null numeric scores")
                print(f"⚠️ Incompatible checkpoint detected ({', '.join(reasons)}).")
                backup_csv = model_out_dir / f"incompatible_checkpoint_backup_{int(time.time())}.csv"
                checkpoint_csv.rename(backup_csv)
                print(f"   Old checkpoint backed up to: {backup_csv.name}")
                completed_qa_ids = set()
                results = []
                print(f"🚀 Starting fresh {REVISION_ID} evaluation on {len(df_input)} samples.")
        except Exception as e:
            print(f"⚠️ Error inspecting checkpoint: {e}. Starting fresh.")
            completed_qa_ids = set()
            results = []
    else:
        completed_qa_ids = set()
        results = []
        print(f"🚀 Starting fresh {REVISION_ID} evaluation on {len(df_input)} samples.")
        
    # Check if already 100% complete
    if len(completed_qa_ids) >= len(df_input):
        print(f"✅ Model [{model_name}] already completed. Proceeding directly to statistical analysis.")
        return pd.DataFrame(results), model_out_dir
        
    print(f"\n📦 Inspecting Configuration for: {model_name}...")
    import transformers
    from transformers import AutoConfig, AutoProcessor
    
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    arch_list = getattr(config, "architectures", [])
    print(f"   Declared Architecture(s): {arch_list} | Model Type: {getattr(config, 'model_type', 'unknown')}")
    
    # Dynamically resolve model class
    model_class = None
    for arch_name in arch_list:
        if hasattr(transformers, arch_name):
            model_class = getattr(transformers, arch_name)
            break
            
    if model_class is None:
        try:
            from transformers import AutoModelForImageTextToText
            model_class = AutoModelForImageTextToText
        except ImportError:
            from transformers import AutoModelForVision2Seq
            model_class = AutoModelForVision2Seq
            
    print(f"   Resolved Model Class: {model_class.__name__}")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True
    )
    
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = model_class.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        trust_remote_code=True
    )
    model.eval()
    target_dev = next(model.parameters()).device
    print(f"✅ Loaded {model_name} ({model_class.__name__}) onto device: {target_dev}")
    
    # Rigorous Empirical Smoke Test on First Pending Sample
    smoke_row = None
    for _, r in df_input.iterrows():
        if r["qa_id"] not in completed_qa_ids:
            smoke_row = r
            break
            
    if smoke_row is not None:
        print("\n" + "=" * 70)
        print("🧪 RUNNING EMPIRICAL SMOKE TEST & CLOCK TIMING (QA ID:", smoke_row["qa_id"], ")")
        print("=" * 70)
        test_img_p = image_map[smoke_row["qa_id"]]
        cid = str(smoke_row["canonical_image_id"])
        
        test_blank_p = PERTURB_DIR / f"blank_{cid}.png"
        if not test_blank_p.exists():
            with Image.open(test_img_p) as orig_pil:
                blank_pil = Image.new("RGB", orig_pil.size, color=(0, 0, 0))
                blank_pil.save(test_blank_p)
            
        p_en = PROMPT_EN_TEMPLATE.format(q=smoke_row["question_en"])
        p_id = PROMPT_ID_TEMPLATE.format(q=smoke_row["question_id"])
        
        t0 = time.time()
        test_pred_en, test_conf_en, test_p_en_yes, test_p_en_no = score_binary_candidates(model, processor, test_img_p, p_en, CANDIDATES_EN, target_dev)
        test_pred_id, test_conf_id, test_p_orig_ya, test_p_orig_tidak = score_binary_candidates(model, processor, test_img_p, p_id, CANDIDATES_ID, target_dev)
        _, _, test_p_blank_ya, test_p_blank_tidak = score_binary_candidates(model, processor, test_blank_p, p_id, CANDIDATES_ID, target_dev)
        t_elapsed = time.time() - t0
        
        test_p_blank_pred = test_p_blank_ya if test_pred_id == "YES" else test_p_blank_tidak
        test_delta_dir = test_conf_id - test_p_blank_pred
        
        print(f"  [Condition P0 - EN] Pred: {test_pred_en} | Conf: {test_conf_en:.4f} (P_yes={test_p_en_yes:.4f}, P_no={test_p_en_no:.4f})")
        print(f"  [Condition P0 - ID] Pred: {test_pred_id} | Conf: {test_conf_id:.4f} (P_ya={test_p_orig_ya:.4f}, P_tidak={test_p_orig_tidak:.4f})")
        print(f"  [Condition P1 - ID] Blank Pred: P_blank(ya)={test_p_blank_ya:.4f}, P_blank(tidak)={test_p_blank_tidak:.4f}")
        print(f"  [Visual Support]   P_blank(pred)={test_p_blank_pred:.4f} -> Delta P_dir = {test_delta_dir:+.4f}")
        print(f"  [Clock Timing]     Measured smoke test runtime: {t_elapsed:.2f} seconds per sample.")
        
        n_rem = len(df_input) - len(completed_qa_ids)
        est_mins = (n_rem * t_elapsed) / 60.0
        print(f"  [Runtime Estimate] Projected time for {n_rem} remaining samples: {est_mins:.1f} minutes ({est_mins/60.0:.2f} hours)")
        print("=" * 70)
        print("✅ Smoke test passed! Tensor shapes, prefix alignment, and timing verified.\n")

    # Main Evaluation Loop
    for idx, row in tqdm(df_input.iterrows(), total=len(df_input), desc=f"Evaluating {model_name.split('/')[-1]}"):
        qa_id = row["qa_id"]
        if qa_id in completed_qa_ids:
            continue
            
        cid = str(row["canonical_image_id"])
        img_id = str(row.get("image_id", cid))
        split_val = str(row.get("split", "test"))
        trans_stat = str(row.get("translation_status", "CLEAN"))
        q_en = str(row["question_en"])
        q_id = str(row["question_id"])
        ref_ans = str(row["reference_answer"]).strip().upper()
        
        orig_img_path = image_map[qa_id]
        
        # 1. Generate Deterministic Black-Blank Image P1
        blank_img_path = PERTURB_DIR / f"blank_{cid}.png"
        if not blank_img_path.exists():
            with Image.open(orig_img_path) as orig_pil:
                blank_pil = Image.new("RGB", orig_pil.size, color=(0, 0, 0))
                blank_pil.save(blank_img_path)
            
        # 2. RQ1: Bilingual Evaluation on P0 (Original)
        prompt_en = PROMPT_EN_TEMPLATE.format(q=q_en)
        pred_en, conf_en, p_en_yes, p_en_no = score_binary_candidates(model, processor, orig_img_path, prompt_en, CANDIDATES_EN, target_dev)
        corr_en = int(pred_en == ref_ans)
        
        prompt_id = PROMPT_ID_TEMPLATE.format(q=q_id)
        pred_id, conf_id, p_orig_ya, p_orig_tidak = score_binary_candidates(model, processor, orig_img_path, prompt_id, CANDIDATES_ID, target_dev)
        corr_id = int(pred_id == ref_ans)
        
        flip = int(pred_en != pred_id)
        joint_corr = int(corr_en and corr_id)
        
        # 3. RQ2: Visual Intervention P1 (Black-Blank) on Indonesian Question
        _, _, p_blank_ya, p_blank_tidak = score_binary_candidates(model, processor, blank_img_path, prompt_id, CANDIDATES_ID, target_dev)
        p_blank_pred = p_blank_ya if pred_id == "YES" else p_blank_tidak
        delta_p_dir = conf_id - p_blank_pred
        delta_p_abs = abs(conf_id - p_blank_pred)
        
        # 4. RQ3: Compute 5 Pre-registered Estimators
        score_m1 = conf_id
        score_m2 = max(0.0, delta_p_dir)
        score_m_dir = conf_id * max(0.0, delta_p_dir)
        score_m_norm = conf_id * (1.0 + delta_p_dir) / 2.0
        score_m3_old = conf_id * (1.0 + delta_p_abs) / 2.0
        
        row_record = {
            "qa_id": qa_id,
            "canonical_image_id": cid,
            "image_id": img_id,
            "split": split_val,
            "translation_status": trans_stat,
            "is_yes_no": bool(row.get("is_yes_no", True)),
            "model_id": model_name,
            "device": str(target_dev),
            "question_en": q_en,
            "question_id": q_id,
            "reference_answer": ref_ans,
            # RQ1 Bilingual
            "pred_en": pred_en,
            "conf_en": conf_en,
            "correct_en": corr_en,
            "p_en_yes": p_en_yes,
            "p_en_no": p_en_no,
            "pred_id": pred_id,
            "conf_id": conf_id,
            "correct_id": corr_id,
            "p_orig_ya": p_orig_ya,
            "p_orig_tidak": p_orig_tidak,
            "flip": flip,
            "joint_correct": joint_corr,
            # RQ2 Visual Intervention
            "p_blank_ya": p_blank_ya,
            "p_blank_tidak": p_blank_tidak,
            "p_blank_pred": p_blank_pred,
            "delta_p_dir": delta_p_dir,
            "delta_p_abs": delta_p_abs,
            # RQ3 Estimators
            "score_m1": score_m1,
            "score_m2": score_m2,
            "score_m_dir": score_m_dir,
            "score_m_norm": score_m_norm,
            "score_m3_old": score_m3_old,
            # Methodological Metadata
            "blank_method": "PIL.Image.new('RGB', size, (0,0,0))",
            "scoring_method": "target_candidate_sequence_scoring",
            "revision_id": REVISION_ID,
            "manifest_sha256": AUTHORITATIVE_MANIFEST_SHA256,
            "timestamp": datetime.now(timezone.utc).isoformat()
        }
        
        results.append(row_record)
        completed_qa_ids.add(qa_id)
        
        # Save checkpoint periodically (every 5 samples)
        if len(results) % 5 == 0 or len(results) == len(df_input):
            pd.DataFrame(results).to_csv(checkpoint_csv, index=False)
            
        # Memory cleanup
        if idx % 10 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    # Final flush to CSV
    res_df = pd.DataFrame(results)
    res_df.to_csv(checkpoint_csv, index=False)
    print(f"💾 Checkpoint saved: {len(res_df)} samples to {checkpoint_csv}")
    
    # Free VRAM before next model
    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 GPU memory released.")
    
    return res_df, model_out_dir

print("✅ Pipeline Engine Defined.")

✅ Pipeline Engine Defined.


In [9]:
# ==============================================================================
# CELL 9: 🚀 RUNNING MODEL 1: Qwen2.5-VL-7B-Instruct
# ==============================================================================
MODEL_1_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

df_results_m1, out_dir_m1 = execute_model_evaluation(
    model_name=MODEL_1_NAME,
    df_input=df_eval,
    image_map=resolved_map
)

print(f"\n📊 Summary Statistics for [{MODEL_1_NAME}] (Full Benchmark N={len(df_results_m1)}):")
print(f"  - English Accuracy      : {df_results_m1['correct_en'].mean()*100:.2f}%")
print(f"  - Indonesian Accuracy   : {df_results_m1['correct_id'].mean()*100:.2f}%")
print(f"  - Language Gap          : {(df_results_m1['correct_en'].mean() - df_results_m1['correct_id'].mean())*100:+.2f}%")
print(f"  - Flip Rate             : {df_results_m1['flip'].mean()*100:.2f}%")
print(f"  - Joint Correctness     : {df_results_m1['joint_correct'].mean()*100:.2f}%")
print(f"  - Mean Delta P (Directed): {df_results_m1['delta_p_dir'].mean():+.4f}")

🚀 Starting fresh rev02 evaluation on 150 samples.

📦 Inspecting Configuration for: Qwen/Qwen2.5-VL-7B-Instruct...


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

   Declared Architecture(s): ['Qwen2_5_VLForConditionalGeneration'] | Model Type: qwen2_5_vl
   Resolved Model Class: Qwen2_5_VLForConditionalGeneration


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

... (8 outputs left out)

✅ Loaded Qwen/Qwen2.5-VL-7B-Instruct (Qwen2_5_VLForConditionalGeneration) onto device: cuda:0

🧪 RUNNING EMPIRICAL SMOKE TEST & CLOCK TIMING (QA ID: 47 )
  [Condition P0 - EN] Pred: YES | Conf: 0.9627 (P_yes=0.9627, P_no=0.0373)
  [Condition P0 - ID] Pred: YES | Conf: 0.9932 (P_ya=0.9932, P_tidak=0.0068)
  [Condition P1 - ID] Blank Pred: P_blank(ya)=0.9739, P_blank(tidak)=0.0261
  [Visual Support]   P_blank(pred)=0.9739 -> Delta P_dir = +0.0193
  [Clock Timing]     Measured smoke test runtime: 30.13 seconds per sample.
  [Runtime Estimate] Projected time for 150 remaining samples: 75.3 minutes (1.26 hours)
✅ Smoke test passed! Tensor shapes, prefix alignment, and timing verified.



Evaluating Qwen2.5-VL-7B-Instruct:   0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


💾 Checkpoint saved: 150 samples to /kaggle/working/results/decisive_validation/rev02/Qwen_Qwen2.5_VL_7B_Instruct/live_validation_checkpoint_Qwen_Qwen2.5_VL_7B_Instruct.csv
🧹 GPU memory released.

📊 Summary Statistics for [Qwen/Qwen2.5-VL-7B-Instruct] (Full Benchmark N=150):
  - English Accuracy      : 78.00%
  - Indonesian Accuracy   : 51.33%
  - Language Gap          : +26.67%
  - Flip Rate             : 37.33%
  - Joint Correctness     : 46.00%
  - Mean Delta P (Directed): +0.0265


In [10]:
# ==============================================================================
# CELL 10: MODEL 1 STATISTICAL ANALYSIS & PLOTTING (TEST-ONLY & SENSITIVITY)
# ==============================================================================
score_cols = ["score_m1", "score_m2", "score_m_dir", "score_m_norm", "score_m3_old"]

def analyze_subset(sub_df, subset_name):
    rows = []
    for sc in score_cols:
        cov, risk, aurc, aurc_trap = compute_discrete_risk_coverage(sub_df[sc].values, sub_df["correct_id"].values)
        r100, _ = get_risk_at_coverage(cov, risk, 1.0)
        r90, _ = get_risk_at_coverage(cov, risk, 0.9)
        r80, _ = get_risk_at_coverage(cov, risk, 0.8)
        r70, _ = get_risk_at_coverage(cov, risk, 0.7)
        r50, _ = get_risk_at_coverage(cov, risk, 0.5)
        rows.append({
            "subset": subset_name,
            "estimator": sc,
            "aurc_discrete": round(aurc, 4),
            "risk_cov_100": round(r100, 4),
            "risk_cov_90": round(r90, 4),
            "risk_cov_80": round(r80, 4),
            "risk_cov_70": round(r70, 4),
            "risk_cov_50": round(r50, 4)
        })
    boot_sum, boot_diff = run_image_cluster_bootstrap(
        sub_df, score_cols, corr_col="correct_id", cluster_col="canonical_image_id", n_boot=1000, seed=42
    )
    for r in rows:
        sc = r["estimator"]
        r["boot_mean"] = round(boot_sum[sc]["mean"], 4)
        r["ci_95"] = f"[{boot_sum[sc]['ci_95_lower']:.4f}, {boot_sum[sc]['ci_95_upper']:.4f}]"
        if sc == "score_m1":
            r["delta_vs_m1"] = 0.0
            r["p_value"] = 1.0
            r["significance"] = "Baseline"
        else:
            diff = boot_diff[sc]
            r["delta_vs_m1"] = round(diff["mean_diff"], 4)
            r["p_value"] = round(diff["p_value"], 4)
            r["significance"] = "Significant (p<0.05)" if diff["p_value"] < 0.05 else "Not Significant"
    return pd.DataFrame(rows)

# 1. Primary Analysis: Test-Only Subset (Exact N=92, 35 clusters)
df_test_m1 = df_results_m1[df_results_m1["split"] == "test"].copy()
assert len(df_test_m1) == 92, f"Expected 92 test samples, got {len(df_test_m1)}"
assert df_test_m1['canonical_image_id'].nunique() == 35, f"Expected 35 test clusters, got {df_test_m1['canonical_image_id'].nunique()}"

print(f"📊 Primary Analysis: Test-Only Subset (N={len(df_test_m1)}, {df_test_m1['canonical_image_id'].nunique()} clusters)")
df_table_test_m1 = analyze_subset(df_test_m1, "Test-Only (N=92)")
display(df_table_test_m1)

# 2. Sensitivity Analysis: Full Benchmark Set (Exact N=150, 58 clusters)
print(f"\n📊 Sensitivity Analysis: Full Benchmark Set (N={len(df_results_m1)}, {df_results_m1['canonical_image_id'].nunique()} clusters)")
df_table_all_m1 = analyze_subset(df_results_m1, "Full-Benchmark (N=150)")
display(df_table_all_m1)

# Combine and save tables
df_final_m1 = pd.concat([df_table_test_m1, df_table_all_m1], ignore_index=True)
comp_path_m1 = out_dir_m1 / "estimator_comparison_qwen7b.csv"
df_final_m1.to_csv(comp_path_m1, index=False)
print(f"💾 Saved estimator comparison: {comp_path_m1}")

# 3. Plot Risk-Coverage Curves for Test-Only Subset
fig_path_m1 = out_dir_m1 / "risk_coverage_curve_qwen7b_test.png"
plot_risk_coverage_curves(df_test_m1, score_cols, fig_path_m1, "Qwen2.5-VL-7B-Instruct (Test-Only N=92)")

📊 Primary Analysis: Test-Only Subset (N=92, 35 clusters)


,subset,estimator,aurc_discrete,risk_cov_100,risk_cov_90,risk_cov_80,risk_cov_70,risk_cov_50,boot_mean,ci_95,delta_vs_m1,p_value,significance
0,Test-Only (N=92),score_m1,0.3146,0.5217,0.4699,0.4189,0.3750,0.3261,0.3102,"[0.1757, 0.4478]",0.0000,1.000,Baseline
1,Test-Only (N=92),score_m2,0.4191,0.5217,0.4699,0.4459,0.4219,0.4348,0.4166,"[0.2504, 0.5698]",0.1063,0.136,Not Significant
2,Test-Only (N=92),score_m_dir,0.4119,0.5217,0.4699,0.4459,0.4219,0.4130,0.4099,"[0.2486, 0.5630]",0.0997,0.146,Not Significant
3,Test-Only (N=92),score_m_norm,0.3263,0.5217,0.4699,0.4054,0.3750,0.3696,0.3230,"[0.1787, 0.4734]",0.0128,0.762,Not Significant
4,Test-Only (N=92),score_m3_old,0.3272,0.5217,0.4699,0.4189,0.3750,0.3696,0.3238,"[0.1784, 0.4728]",0.0136,0.734,Not Significant



📊 Sensitivity Analysis: Full Benchmark Set (N=150, 58 clusters)


,subset,estimator,aurc_discrete,risk_cov_100,risk_cov_90,risk_cov_80,risk_cov_70,risk_cov_50,boot_mean,ci_95,delta_vs_m1,p_value,significance
0,Full-Benchmark (N=150),score_m1,0.2782,0.4867,0.4519,0.4000,0.3524,0.2667,0.2770,"[0.1756, 0.3900]",0.0000,1.000,Baseline
1,Full-Benchmark (N=150),score_m2,0.3964,0.4867,0.4593,0.4167,0.3905,0.4267,0.3936,"[0.2688, 0.5116]",0.1166,0.012,Significant (p<0.05)
2,Full-Benchmark (N=150),score_m_dir,0.3912,0.4867,0.4593,0.4167,0.3905,0.4267,0.3886,"[0.2630, 0.5050]",0.1116,0.018,Significant (p<0.05)
3,Full-Benchmark (N=150),score_m_norm,0.2879,0.4867,0.4370,0.3917,0.3333,0.3200,0.2874,"[0.1789, 0.3979]",0.0104,0.740,Not Significant
4,Full-Benchmark (N=150),score_m3_old,0.2883,0.4867,0.4444,0.3917,0.3429,0.3200,0.2878,"[0.1783, 0.4005]",0.0108,0.730,Not Significant


💾 Saved estimator comparison: /kaggle/working/results/decisive_validation/rev02/Qwen_Qwen2.5_VL_7B_Instruct/estimator_comparison_qwen7b.csv
📊 Saved Risk-Coverage curve: /kaggle/working/results/decisive_validation/rev02/Qwen_Qwen2.5_VL_7B_Instruct/risk_coverage_curve_qwen7b_test.png


In [11]:
# ==============================================================================
# CELL 11: 🚀 RUNNING MODEL 2: HuatuoGPT-Vision-7B
# ==============================================================================
MODEL_2_NAME = "FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL"

df_results_m2, out_dir_m2 = execute_model_evaluation(
    model_name=MODEL_2_NAME,
    df_input=df_eval,
    image_map=resolved_map
)

print(f"\n📊 Summary Statistics for [{MODEL_2_NAME}] (Full Benchmark N={len(df_results_m2)}):")
print(f"  - English Accuracy      : {df_results_m2['correct_en'].mean()*100:.2f}%")
print(f"  - Indonesian Accuracy   : {df_results_m2['correct_id'].mean()*100:.2f}%")
print(f"  - Language Gap          : {(df_results_m2['correct_en'].mean() - df_results_m2['correct_id'].mean())*100:+.2f}%")
print(f"  - Flip Rate             : {df_results_m2['flip'].mean()*100:.2f}%")
print(f"  - Joint Correctness     : {df_results_m2['joint_correct'].mean()*100:.2f}%")
print(f"  - Mean Delta P (Directed): {df_results_m2['delta_p_dir'].mean():+.4f}")

🚀 Starting fresh rev02 evaluation on 150 samples.

📦 Inspecting Configuration for: FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL...


config.json: reconstructing file:   0%|          |  0.00B / 1.50kB            

config.json: downloading bytes:           |  0.00B            

   Declared Architecture(s): ['Qwen2_5_VLForConditionalGeneration'] | Model Type: qwen2_5_vl
   Resolved Model Class: Qwen2_5_VLForConditionalGeneration


preprocessor_config.json: reconstructing file:   0%|          |  0.00B /   575B            

preprocessor_config.json: downloading bytes:           |  0.00B            

... (20 outputs left out)

✅ Loaded FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL (Qwen2_5_VLForConditionalGeneration) onto device: cuda:0

🧪 RUNNING EMPIRICAL SMOKE TEST & CLOCK TIMING (QA ID: 47 )
  [Condition P0 - EN] Pred: YES | Conf: 0.9149 (P_yes=0.9149, P_no=0.0851)
  [Condition P0 - ID] Pred: YES | Conf: 0.9947 (P_ya=0.9947, P_tidak=0.0053)
  [Condition P1 - ID] Blank Pred: P_blank(ya)=0.9579, P_blank(tidak)=0.0421
  [Visual Support]   P_blank(pred)=0.9579 -> Delta P_dir = +0.0368
  [Clock Timing]     Measured smoke test runtime: 32.36 seconds per sample.
  [Runtime Estimate] Projected time for 150 remaining samples: 80.9 minutes (1.35 hours)
✅ Smoke test passed! Tensor shapes, prefix alignment, and timing verified.



Evaluating HuatuoGPT-Vision-7B-Qwen2.5VL:   0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


💾 Checkpoint saved: 150 samples to /kaggle/working/results/decisive_validation/rev02/FreedomIntelligence_HuatuoGPT_Vision_7B_Qwen2.5VL/live_validation_checkpoint_FreedomIntelligence_HuatuoGPT_Vision_7B_Qwen2.5VL.csv
🧹 GPU memory released.

📊 Summary Statistics for [FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL] (Full Benchmark N=150):
  - English Accuracy      : 66.67%
  - Indonesian Accuracy   : 47.33%
  - Language Gap          : +19.33%
  - Flip Rate             : 22.00%
  - Joint Correctness     : 46.00%
  - Mean Delta P (Directed): +0.0233


In [12]:
# ==============================================================================
# CELL 12: MODEL 2 STATISTICAL ANALYSIS & PLOTTING (TEST-ONLY & SENSITIVITY)
# ==============================================================================
df_test_m2 = df_results_m2[df_results_m2["split"] == "test"].copy()
assert len(df_test_m2) == 92, f"Expected 92 test samples, got {len(df_test_m2)}"
assert df_test_m2['canonical_image_id'].nunique() == 35, f"Expected 35 test clusters, got {df_test_m2['canonical_image_id'].nunique()}"

print(f"📊 Primary Analysis: Test-Only Subset (N={len(df_test_m2)}, {df_test_m2['canonical_image_id'].nunique()} clusters)")
df_table_test_m2 = analyze_subset(df_test_m2, "Test-Only (N=92)")
display(df_table_test_m2)

print(f"\n📊 Sensitivity Analysis: Full Benchmark Set (N={len(df_results_m2)}, {df_results_m2['canonical_image_id'].nunique()} clusters)")
df_table_all_m2 = analyze_subset(df_results_m2, "Full-Benchmark (N=150)")
display(df_table_all_m2)

df_final_m2 = pd.concat([df_table_test_m2, df_table_all_m2], ignore_index=True)
comp_path_m2 = out_dir_m2 / "estimator_comparison_huatuo7b.csv"
df_final_m2.to_csv(comp_path_m2, index=False)
print(f"💾 Saved estimator comparison: {comp_path_m2}")

fig_path_m2 = out_dir_m2 / "risk_coverage_curve_huatuo7b_test.png"
plot_risk_coverage_curves(df_test_m2, score_cols, fig_path_m2, "HuatuoGPT-Vision-7B (Test-Only N=92)")

📊 Primary Analysis: Test-Only Subset (N=92, 35 clusters)


,subset,estimator,aurc_discrete,risk_cov_100,risk_cov_90,risk_cov_80,risk_cov_70,risk_cov_50,boot_mean,ci_95,delta_vs_m1,p_value,significance
0,Test-Only (N=92),score_m1,0.3584,0.5435,0.4940,0.4595,0.4219,0.3261,0.3540,"[0.2158, 0.4879]",0.0000,1.000,Baseline
1,Test-Only (N=92),score_m2,0.5182,0.5435,0.5422,0.5405,0.5312,0.5435,0.5121,"[0.3493, 0.6514]",0.1580,0.012,Significant (p<0.05)
2,Test-Only (N=92),score_m_dir,0.5041,0.5435,0.5422,0.5405,0.5312,0.5435,0.4972,"[0.3321, 0.6411]",0.1431,0.018,Significant (p<0.05)
3,Test-Only (N=92),score_m_norm,0.4048,0.5435,0.5181,0.4865,0.4688,0.3913,0.4031,"[0.2491, 0.5557]",0.0491,0.228,Not Significant
4,Test-Only (N=92),score_m3_old,0.4046,0.5435,0.5181,0.4865,0.4688,0.3913,0.4029,"[0.2491, 0.5556]",0.0488,0.228,Not Significant



📊 Sensitivity Analysis: Full Benchmark Set (N=150, 58 clusters)


,subset,estimator,aurc_discrete,risk_cov_100,risk_cov_90,risk_cov_80,risk_cov_70,risk_cov_50,boot_mean,ci_95,delta_vs_m1,p_value,significance
0,Full-Benchmark (N=150),score_m1,0.3126,0.5267,0.4741,0.4250,0.3810,0.3067,0.3120,"[0.2057, 0.4245]",0.0000,1.000,Baseline
1,Full-Benchmark (N=150),score_m2,0.5120,0.5267,0.4889,0.5167,0.5048,0.5067,0.5057,"[0.3769, 0.6171]",0.1937,0.001,Significant (p<0.05)
2,Full-Benchmark (N=150),score_m_dir,0.5031,0.5267,0.4889,0.5167,0.4952,0.5067,0.4967,"[0.3659, 0.6089]",0.1846,0.001,Significant (p<0.05)
3,Full-Benchmark (N=150),score_m_norm,0.3859,0.5267,0.4815,0.4500,0.4095,0.3467,0.3854,"[0.2633, 0.5074]",0.0733,0.026,Significant (p<0.05)
4,Full-Benchmark (N=150),score_m3_old,0.3858,0.5267,0.4741,0.4500,0.4095,0.3467,0.3852,"[0.2633, 0.5071]",0.0732,0.026,Significant (p<0.05)


💾 Saved estimator comparison: /kaggle/working/results/decisive_validation/rev02/FreedomIntelligence_HuatuoGPT_Vision_7B_Qwen2.5VL/estimator_comparison_huatuo7b.csv
📊 Saved Risk-Coverage curve: /kaggle/working/results/decisive_validation/rev02/FreedomIntelligence_HuatuoGPT_Vision_7B_Qwen2.5VL/risk_coverage_curve_huatuo7b_test.png


In [13]:
# ==============================================================================
# CELL 13: 📊 FINAL THESIS COMPARATIVE LEADERBOARD (RQ1 - RQ3)
# ==============================================================================
# Primary evaluations strictly on Test-Only (N=92, 35 clusters)
t1 = df_results_m1[df_results_m1["split"] == "test"]
t2 = df_results_m2[df_results_m2["split"] == "test"]

assert len(t1) == 92, f"Model 1 test count must be 92, got {len(t1)}"
assert len(t2) == 92, f"Model 2 test count must be 92, got {len(t2)}"

st1 = df_table_test_m1.set_index("estimator")
st2 = df_table_test_m2.set_index("estimator")

comparative_rows = [
    {
        "Research Question": "RQ1: English Accuracy (Test-Only N=92)",
        "Qwen2.5-VL-7B": f"{t1['correct_en'].mean()*100:.2f}%",
        "HuatuoGPT-Vision-7B": f"{t2['correct_en'].mean()*100:.2f}%",
        "Interpretation": "Performa pada radiograf asli dalam bahasa sumber (EN)"
    },
    {
        "Research Question": "RQ1: Indonesian Accuracy (Test-Only N=92)",
        "Qwen2.5-VL-7B": f"{t1['correct_id'].mean()*100:.2f}%",
        "HuatuoGPT-Vision-7B": f"{t2['correct_id'].mean()*100:.2f}%",
        "Interpretation": "Performa pada radiograf asli di bawah translasi (ID)"
    },
    {
        "Research Question": "RQ1: Language Gap (EN - ID)",
        "Qwen2.5-VL-7B": f"{(t1['correct_en'].mean() - t1['correct_id'].mean())*100:+.2f}%",
        "HuatuoGPT-Vision-7B": f"{(t2['correct_en'].mean() - t2['correct_id'].mean())*100:+.2f}%",
        "Interpretation": "Penurunan akurasi akibat translasi bahasa (positif = degradasi)"
    },
    {
        "Research Question": "RQ1: Cross-Lingual Flip Rate",
        "Qwen2.5-VL-7B": f"{t1['flip'].mean()*100:.2f}%",
        "HuatuoGPT-Vision-7B": f"{t2['flip'].mean()*100:.2f}%",
        "Interpretation": "Proporsi sampel di mana jawaban model berubah antara EN dan ID"
    },
    {
        "Research Question": "RQ1: Joint Bilingual Correctness",
        "Qwen2.5-VL-7B": f"{t1['joint_correct'].mean()*100:.2f}%",
        "HuatuoGPT-Vision-7B": f"{t2['joint_correct'].mean()*100:.2f}%",
        "Interpretation": "Konsistensi menjawab benar pada kedua bahasa sekaligus"
    },
    {
        "Research Question": "RQ2: Visual Sensitivity (Mean Delta P_dir)",
        "Qwen2.5-VL-7B": f"{t1['delta_p_dir'].mean():+.4f}",
        "HuatuoGPT-Vision-7B": f"{t2['delta_p_dir'].mean():+.4f}",
        "Interpretation": "Dukungan visual terarah (P0 Original vs P1 Black-Blank)"
    },
    {
        "Research Question": "RQ3: M1 Confidence-Only AURC",
        "Qwen2.5-VL-7B": f"{st1.loc['score_m1', 'aurc_discrete']:.4f}",
        "HuatuoGPT-Vision-7B": f"{st2.loc['score_m1', 'aurc_discrete']:.4f}",
        "Interpretation": "Baseline selective risk menggunakan kepercayaan asli"
    },
    {
        "Research Question": "RQ3: M_dir Directed Multiplicative AURC",
        "Qwen2.5-VL-7B": f"{st1.loc['score_m_dir', 'aurc_discrete']:.4f} (p={st1.loc['score_m_dir', 'p_value']:.4f})",
        "HuatuoGPT-Vision-7B": f"{st2.loc['score_m_dir', 'aurc_discrete']:.4f} (p={st2.loc['score_m_dir', 'p_value']:.4f})",
        "Interpretation": "Estimator utama terdaftar: conf * max(0, Delta)"
    },
    {
        "Research Question": "RQ3: M_norm Normalized Linear AURC",
        "Qwen2.5-VL-7B": f"{st1.loc['score_m_norm', 'aurc_discrete']:.4f} (p={st1.loc['score_m_norm', 'p_value']:.4f})",
        "HuatuoGPT-Vision-7B": f"{st2.loc['score_m_norm', 'aurc_discrete']:.4f} (p={st2.loc['score_m_norm', 'p_value']:.4f})",
        "Interpretation": "Estimator linear terarah: conf * (1 + Delta) / 2"
    },
    {
        "Research Question": "RQ3: M2 Visual-Only AURC",
        "Qwen2.5-VL-7B": f"{st1.loc['score_m2', 'aurc_discrete']:.4f} (p={st1.loc['score_m2', 'p_value']:.4f})",
        "HuatuoGPT-Vision-7B": f"{st2.loc['score_m2', 'aurc_discrete']:.4f} (p={st2.loc['score_m2', 'p_value']:.4f})",
        "Interpretation": "Estimator visual terisolasi: max(0, Delta)"
    },
    {
        "Research Question": "RQ3: M3_old Historical Unsigned AURC",
        "Qwen2.5-VL-7B": f"{st1.loc['score_m3_old', 'aurc_discrete']:.4f} (p={st1.loc['score_m3_old', 'p_value']:.4f})",
        "HuatuoGPT-Vision-7B": f"{st2.loc['score_m3_old', 'aurc_discrete']:.4f} (p={st2.loc['score_m3_old', 'p_value']:.4f})",
        "Interpretation": "Baseline lama tidak terarah: conf * (1 + |Delta|) / 2"
    }
]

df_comparative = pd.DataFrame(comparative_rows)
print("\n" + "=" * 90)
print("🏆 FINAL EMPIRICAL COMPARISON TABLE (TEST-ONLY N=92, 35 CLUSTERS)")
print("=" * 90)
display(df_comparative)

comp_save_path = RESULTS_DIR / "final_cross_model_comparison_test_only.csv"
df_comparative.to_csv(comp_save_path, index=False)
print(f"\n💾 Saved final cross-model comparative table to: {comp_save_path}")


🏆 FINAL EMPIRICAL COMPARISON TABLE (TEST-ONLY N=92, 35 CLUSTERS)


,Research Question,Qwen2.5-VL-7B,HuatuoGPT-Vision-7B,Interpretation
0,RQ1: English Accuracy (Test-Only N=92),75.00%,64.13%,Performa pada radiograf asli dalam bahasa sumb...
1,RQ1: Indonesian Accuracy (Test-Only N=92),47.83%,45.65%,Performa pada radiograf asli di bawah translas...
2,RQ1: Language Gap (EN - ID),+27.17%,+18.48%,Penurunan akurasi akibat translasi bahasa (pos...
3,RQ1: Cross-Lingual Flip Rate,38.04%,20.65%,Proporsi sampel di mana jawaban model berubah ...
4,RQ1: Joint Bilingual Correctness,42.39%,44.57%,Konsistensi menjawab benar pada kedua bahasa s...
5,RQ2: Visual Sensitivity (Mean Delta P_dir),+0.0153,+0.0263,Dukungan visual terarah (P0 Original vs P1 Bla...
6,RQ3: M1 Confidence-Only AURC,0.3146,0.3584,Baseline selective risk menggunakan kepercayaa...
7,RQ3: M_dir Directed Multiplicative AURC,0.4119 (p=0.1460),0.5041 (p=0.0180),"Estimator utama terdaftar: conf * max(0, Delta)"
8,RQ3: M_norm Normalized Linear AURC,0.3263 (p=0.7620),0.4048 (p=0.2280),Estimator linear terarah: conf * (1 + Delta) / 2
9,RQ3: M2 Visual-Only AURC,0.4191 (p=0.1360),0.5182 (p=0.0120),"Estimator visual terisolasi: max(0, Delta)"



💾 Saved final cross-model comparative table to: /kaggle/working/results/decisive_validation/rev02/final_cross_model_comparison_test_only.csv


In [14]:
# ==============================================================================
# CELL 14: PACKAGING & ARCHIVING RESULTS (REV02)
# ==============================================================================
import shutil
import zipfile
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
bundle_zip_name = f"decisive_validation_rev02_results_{timestamp}.zip"
bundle_zip_path = WORKSPACE / bundle_zip_name

print(f"📦 Packaging all rev02 results from {RESULTS_DIR} into: {bundle_zip_path}...")
with zipfile.ZipFile(bundle_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(RESULTS_DIR):
        for file in files:
            full_p = Path(root) / file
            rel_p = full_p.relative_to(RESULTS_DIR)
            zipf.write(full_p, arcname=str(Path("rev02") / rel_p))

zip_size_mb = bundle_zip_path.stat().st_size / (1024**2)
print(f"✅ Created downloadable archive: {bundle_zip_name} ({zip_size_mb:.2f} MB)")
print(f"📍 Location on Kaggle: {bundle_zip_path.resolve()}")

# Also create fixed-name copy for convenience
fixed_zip = WORKSPACE / "decisive_validation_rev02_latest.zip"
shutil.copy2(bundle_zip_path, fixed_zip)

try:
    from IPython.display import FileLink, display
    print("\n🔗 Clickable Download Links (Direct Browser Download):")
    display(FileLink(bundle_zip_name))
    display(FileLink("decisive_validation_rev02_latest.zip"))
except Exception as e:
    pass

print("\n💡 PANDUAN MENGUNDUH SELURUH FILE OUTPUT DI KAGGLE:")
print("1. Cara 1 (Langsung): Klik link 'FileLink' di atas untuk mengunduh langsung dari browser.")
print("2. Cara 2 (Output Sidebar): Di panel sebelah kanan layar Kaggle, buka tab 'Output'.")
print(f"   - Temukan '{bundle_zip_name}' atau 'decisive_validation_rev02_latest.zip'.")
print("   - Klik tombol titik tiga (...) di samping file, lalu pilih 'Download'.")
print("   - Anda juga dapat mengunduh individual file CSV atau PNG di dalam folder 'results/'.")
print("3. Ekstrak file zip ke direktori lokal proyek:")
print("   c:\\.PUSAT\\Project\\Kuliah\\VLM\\results\\decisive_validation\\")

📦 Packaging all rev02 results from /kaggle/working/results/decisive_validation/rev02 into: /kaggle/working/decisive_validation_rev02_results_20261019_175949.zip...
✅ Created downloadable archive: decisive_validation_rev02_results_20261019_175949.zip (0.89 MB)
📍 Location on Kaggle: /kaggle/working/decisive_validation_rev02_results_20261019_175949.zip

🔗 Clickable Download Links (Direct Browser Download):


/kaggle/working/decisive_validation_rev02_results_20261019_175949.zip

/kaggle/working/decisive_validation_rev02_latest.zip


💡 PANDUAN MENGUNDUH SELURUH FILE OUTPUT DI KAGGLE:
1. Cara 1 (Langsung): Klik link 'FileLink' di atas untuk mengunduh langsung dari browser.
2. Cara 2 (Output Sidebar): Di panel sebelah kanan layar Kaggle, buka tab 'Output'.
   - Temukan 'decisive_validation_rev02_results_20261019_175949.zip' atau 'decisive_validation_rev02_latest.zip'.
   - Klik tombol titik tiga (...) di samping file, lalu pilih 'Download'.
   - Anda juga dapat mengunduh individual file CSV atau PNG di dalam folder 'results/'.
3. Ekstrak file zip ke direktori lokal proyek:
   c:\.PUSAT\Project\Kuliah\VLM\results\decisive_validation\
